# Time-aware forecasting of firm-level credit demand 
This notebook keeps the earlier report's focused comparison of **regularized linear models** and **XGBoost**, but replaces the weaker parts of its validation and tuning design.

Key methodological changes:

- expanding-window validation remains strictly inside the 2010–2019 development period;
- any computational subsampling is performed at the **firm level**, retaining each sampled firm's full history;
- for regularized linear models, winsorization, imputation, scaling, and one-hot encoding are fitted on training folds only; XGBoost uses native missing-value handling and unscaled engineered features;
- one-hot indicators are standardized before penalized regression;
- Ridge, Lasso, and Elastic Net are compared using the **one-standard-error rule**, with an explicit convergence audit;
- the selected linear family and XGBoost are **retuned separately for every feature block**;
- XGBoost tunes a fixed `n_estimators` value through time-aware CV and does not use the same validation fold for early stopping and scoring;
- the model/feature specification is frozen from development-period evidence before the held-out test is evaluated;
- thin/incomplete test years are reported separately rather than pooled into the headline result;
- incremental survey/text comparisons use the year as the effective unit of variation;
- optional robustness sections assess linear-family choice by information block, XGBoost grid boundaries, fold-fitted clipping for XGBoost, performance for previously seen versus new firms, and heterogeneity across firm characteristics.

## Final run

The notebook defaults to `QUICK_MODE = False`. Because the robustness sections depend on objects created throughout the workflow, restart the kernel and run all cells once. Do not change the specification after the held-out results appear.


## 1. Imports and configuration

In [2]:
from __future__ import annotations

import gc
import json
import os
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.model_selection import GridSearchCV, ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import xgboost as xgb

warnings.filterwarnings("ignore", category=FutureWarning)
# ConvergenceWarning is deliberately not suppressed globally; it is audited below.
pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 200)

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
DATA_PATH = Path("data/model_input/analysis_panel.parquet")
OUTPUT_DIR = Path("outputs/time_aware_model_report_lecture_aligned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Evaluation design
# -----------------------------------------------------------------------------
RANDOM_STATE = 42
YEAR_COL = "closdate_year"
ID_COL = "idnr"
TARGET = "ncliGrowthNextYear"

TEST_START_YEAR = 2020
FIRST_VALIDATION_YEAR = 2015
LAST_VALIDATION_YEAR = 2019
MIN_YEAR_OBSERVATIONS = 10_000

# Numeric clipping is learned inside every linear-model training fold.
WINSOR_LOWER = 0.01
WINSOR_UPPER = 0.99
LINEAR_MAX_ITER = 10_000

# -----------------------------------------------------------------------------
# Runtime controls
# -----------------------------------------------------------------------------
QUICK_MODE = False

# Sampling is at firm level: all available years of a selected firm are retained.
SCREEN_ROWS_PER_YEAR = 8_000 if QUICK_MODE else 20_000
TUNING_ROWS_PER_YEAR = 15_000 if QUICK_MODE else 50_000
FINAL_ROWS_PER_YEAR = 15_000 if QUICK_MODE else None  # None = full development sample

XGB_SCREEN_TRIALS = 8 if QUICK_MODE else 20
XGB_REFINE_TOP_K = 3 if QUICK_MODE else 5
N_BOOTSTRAP = 500 if QUICK_MODE else 2_000
PERMUTATION_SAMPLE_SIZE = 10_000 if QUICK_MODE else 50_000
PERMUTATION_REPEATS = 3

N_JOBS = max(1, min(4, os.cpu_count() or 1))
LINEAR_SEARCH_N_JOBS = 1  # avoids duplicating the large panel in RAM

RETUNE_PER_BLOCK = True
RUN_FINAL_TEST = True
RUN_PERMUTATION_IMPORTANCE = True
SAVE_MODELS = True  # export the frozen model and metadata after the definitive run

# Optional robustness analyses. The refit-based checks add runtime but never use
# held-out outcomes to change the frozen primary specification.
RUN_OPTIONAL_REFITS = not QUICK_MODE
RUN_LINEAR_FAMILY_BY_BLOCK_ROBUSTNESS = RUN_OPTIONAL_REFITS
RUN_XGB_BOUNDARY_SENSITIVITY = RUN_OPTIONAL_REFITS
RUN_CLIPPED_XGB_ROBUSTNESS = RUN_OPTIONAL_REFITS

# These analyses reuse predictions from the primary run and are comparatively cheap.
RUN_HELDOUT_INCREMENT_TESTS = True
RUN_HETEROGENEITY_ANALYSIS = True
RUN_SEEN_NEW_FIRM_ROBUSTNESS = True
RUN_MACRO_SENSITIVITY = True

print(json.dumps({
    "quick_mode": QUICK_MODE,
    "screen_rows_per_year": SCREEN_ROWS_PER_YEAR,
    "tuning_rows_per_year": TUNING_ROWS_PER_YEAR,
    "final_rows_per_year": FINAL_ROWS_PER_YEAR,
    "xgb_screen_trials": XGB_SCREEN_TRIALS,
    "xgb_refine_top_k": XGB_REFINE_TOP_K,
    "threads": N_JOBS,
    "run_final_test": RUN_FINAL_TEST,
    "run_optional_refits": RUN_OPTIONAL_REFITS,
    "linear_family_by_block_robustness": RUN_LINEAR_FAMILY_BY_BLOCK_ROBUSTNESS,
    "xgb_boundary_sensitivity": RUN_XGB_BOUNDARY_SENSITIVITY,
    "clipped_xgb_robustness": RUN_CLIPPED_XGB_ROBUSTNESS,
    "heterogeneity_analysis": RUN_HETEROGENEITY_ANALYSIS,
    "seen_new_firm_robustness": RUN_SEEN_NEW_FIRM_ROBUSTNESS,
    "macro_sensitivity": RUN_MACRO_SENSITIVITY,
}, indent=2))


{
  "quick_mode": false,
  "screen_rows_per_year": 20000,
  "tuning_rows_per_year": 50000,
  "final_rows_per_year": null,
  "xgb_screen_trials": 20,
  "xgb_refine_top_k": 5,
  "threads": 4,
  "run_final_test": true,
  "run_optional_refits": true,
  "linear_family_by_block_robustness": true,
  "xgb_boundary_sensitivity": true,
  "clipped_xgb_robustness": true,
  "heterogeneity_analysis": true,
  "seen_new_firm_robustness": true,
  "macro_sensitivity": true
}


### Computational note

The full-data final refit uses all eligible pre-2020 observations because `QUICK_MODE=False` sets
`FINAL_ROWS_PER_YEAR=None`. Hyperparameter search still uses deterministic firm-level samples to
keep the expanding-window search computationally feasible. This does not turn the final model into
a sample-trained model: only the tuning stage is subsampled.

`RUN_OPTIONAL_REFITS=True` activates the block-specific linear-family check, the targeted XGBoost
boundary sensitivity, and the clipped-XGBoost robustness check. These use development information
only for tuning and add substantial runtime. Set `QUICK_MODE=True` for a smoke test; restore
`QUICK_MODE=False` and run the notebook from the top for the reportable final results.


## 2. Reusable validation and preprocessing utilities

In [3]:
class QuantileWinsorizer(BaseEstimator, TransformerMixin):
    """Clip numeric features using quantiles learned on the training data only."""

    def __init__(self, lower_quantile: float = 0.01, upper_quantile: float = 0.99):
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile

    @staticmethod
    def _as_array(X) -> np.ndarray:
        if hasattr(X, "to_numpy"):
            return X.to_numpy(dtype=np.float32, copy=True)
        return np.asarray(X, dtype=np.float32)

    def fit(self, X, y=None):
        values = self._as_array(X)
        self.n_features_in_ = values.shape[1]
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            lower = np.nanquantile(values, self.lower_quantile, axis=0)
            upper = np.nanquantile(values, self.upper_quantile, axis=0)
        self.lower_bounds_ = np.where(np.isfinite(lower), lower, -np.inf).astype(np.float32)
        self.upper_bounds_ = np.where(np.isfinite(upper), upper, np.inf).astype(np.float32)
        return self

    def transform(self, X):
        return np.clip(self._as_array(X), self.lower_bounds_, self.upper_bounds_)

    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return np.asarray(input_features, dtype=object)
        if hasattr(self, "feature_names_in_"):
            return self.feature_names_in_
        return np.asarray([f"x{i}" for i in range(self.n_features_in_)], dtype=object)


def firm_hash_uniform(ids: pd.Series) -> np.ndarray:
    """Stable pseudo-uniform number for deterministic firm-level sampling."""
    hashed = pd.util.hash_pandas_object(ids.astype(str), index=False).to_numpy(dtype="uint64")
    return (hashed % np.uint64(10_000_019)).astype(np.float64) / 10_000_019.0


def sample_firms_to_target(frame: pd.DataFrame, target_rows_per_year: int | None) -> pd.DataFrame:
    """Sample firms, not rows, and retain the complete history of every selected firm."""
    if target_rows_per_year is None or frame.empty:
        return frame.copy()
    busiest_year = int(frame[YEAR_COL].value_counts().max())
    if busiest_year <= target_rows_per_year:
        return frame.copy()
    fraction = target_rows_per_year / busiest_year
    sampled = frame.loc[firm_hash_uniform(frame[ID_COL]) < fraction].copy()
    return sampled.sort_values([YEAR_COL, ID_COL]).reset_index(drop=True)


def expanding_window_folds(
    frame: pd.DataFrame,
    first_validation_year: int,
    last_validation_year: int,
) -> tuple[list[tuple[np.ndarray, np.ndarray]], list[int]]:
    """Train on all earlier years and validate on one later year."""
    splits: list[tuple[np.ndarray, np.ndarray]] = []
    years: list[int] = []
    for validation_year in range(first_validation_year, last_validation_year + 1):
        train_idx = np.flatnonzero(frame[YEAR_COL].lt(validation_year).to_numpy())
        validation_idx = np.flatnonzero(frame[YEAR_COL].eq(validation_year).to_numpy())
        if len(train_idx) == 0 or len(validation_idx) == 0:
            continue
        if int(frame.iloc[train_idx][YEAR_COL].max()) >= validation_year:
            raise AssertionError(f"Look-ahead detected in validation year {validation_year}.")
        splits.append((train_idx, validation_idx))
        years.append(validation_year)
    if not splits:
        raise ValueError("No expanding-window folds could be constructed.")
    return splits, years


def regression_metrics(y_true, y_pred, baseline_pred, years=None) -> dict[str, float]:
    """RMSE, MAE, and OOS R² against the actual benchmark predictions."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    bp = np.asarray(baseline_pred, dtype=float)
    if bp.ndim == 0:
        bp = np.full(len(yt), float(bp), dtype=float)
    valid = np.isfinite(yt) & np.isfinite(yp) & np.isfinite(bp)
    yt, yp, bp = yt[valid], yp[valid], bp[valid]
    if len(yt) == 0:
        return {
            "RMSE": np.nan,
            "MAE": np.nan,
            "R2_oos": np.nan,
            "R2_oos_within_year": np.nan,
            "n": 0,
        }

    sse = float(np.sum((yt - yp) ** 2))
    benchmark_sse = float(np.sum((yt - bp) ** 2))

    within_year = np.nan
    if years is not None:
        yr = np.asarray(years)[valid]
        year_means = pd.Series(yt).groupby(pd.Series(yr)).transform("mean").to_numpy()
        within_sse = float(np.sum((yt - year_means) ** 2))
        if within_sse > 0:
            within_year = float(1 - sse / within_sse)

    return {
        "RMSE": float(np.sqrt(sse / len(yt))),
        "MAE": float(np.mean(np.abs(yt - yp))),
        "R2_oos": float(1 - sse / benchmark_sse) if benchmark_sse > 0 else np.nan,
        "R2_oos_within_year": within_year,
        "n": int(len(yt)),
    }


def select_one_standard_error(
    cv_results: dict,
    complexity_key: str,
    n_splits: int,
) -> tuple[dict, dict]:
    """Choose the strongest regularization within one SE of minimum CV RMSE."""
    results = pd.DataFrame(cv_results).reset_index(drop=True)
    best_index = int(results["mean_test_score"].idxmax())
    best_score = float(results.loc[best_index, "mean_test_score"])
    one_se = float(results.loc[best_index, "std_test_score"]) / np.sqrt(n_splits)
    eligible = results[results["mean_test_score"] >= best_score - one_se].copy()
    eligible["alpha_order"] = eligible["params"].apply(lambda p: float(p[complexity_key]))
    eligible["l1_order"] = eligible["params"].apply(
        lambda p: float(p.get("model__l1_ratio", 0.0))
    )
    chosen = eligible.sort_values(["alpha_order", "l1_order"], ascending=False).iloc[0]
    return dict(chosen["params"]), {
        "minimum_RMSE": -best_score,
        "one_se": one_se,
        "chosen_RMSE": -float(chosen["mean_test_score"]),
        "n_within_one_se": int(len(eligible)),
        "minimum_parameters": results.loc[best_index, "params"],
    }


def check_grid_boundaries(selected: dict, grid: dict, label: str) -> list[str]:
    """Flag selected numeric values at the edge of the searched grid."""
    messages: list[str] = []
    for key, value in selected.items():
        if key not in grid:
            continue
        values = list(grid[key])
        if not values or not all(isinstance(v, (int, float, np.number)) for v in values):
            continue
        minimum, maximum = min(values), max(values)
        if np.isclose(float(value), float(minimum)):
            messages.append(f"{label}: {key}={value} is at the LOWER grid boundary.")
        if np.isclose(float(value), float(maximum)) and not np.isclose(minimum, maximum):
            messages.append(f"{label}: {key}={value} is at the UPPER grid boundary.")
    return messages


def year_cluster_forecast_test(y_true, pred_a, pred_b, years, loss="squared") -> dict:
    """Compare forecasts using one mean loss differential per evaluation year."""
    yt, pa, pb = (np.asarray(v, dtype=float) for v in (y_true, pred_a, pred_b))
    yr = np.asarray(years)
    valid = np.isfinite(yt) & np.isfinite(pa) & np.isfinite(pb)
    yt, pa, pb, yr = yt[valid], pa[valid], pb[valid], yr[valid]
    if loss == "squared":
        differential = (yt - pa) ** 2 - (yt - pb) ** 2
    elif loss == "absolute":
        differential = np.abs(yt - pa) - np.abs(yt - pb)
    else:
        raise ValueError("loss must be 'squared' or 'absolute'.")

    year_means = pd.DataFrame({"d": differential, "year": yr}).groupby("year")["d"].mean()
    g = len(year_means)
    point = float(year_means.mean()) if g else np.nan
    if g < 2:
        return {"mean_loss_difference": point, "t_stat": np.nan, "p_value": np.nan,
                "ci_low": np.nan, "ci_high": np.nan, "n_years": g}
    standard_error = float(year_means.std(ddof=1) / np.sqrt(g))
    if standard_error == 0:
        return {"mean_loss_difference": point, "t_stat": np.nan, "p_value": np.nan,
                "ci_low": point, "ci_high": point, "n_years": g}
    t_stat = point / standard_error
    critical = float(stats.t.ppf(0.975, df=g - 1))
    return {
        "mean_loss_difference": point,
        "t_stat": float(t_stat),
        "p_value": float(2 * stats.t.sf(abs(t_stat), df=g - 1)),
        "ci_low": point - critical * standard_error,
        "ci_high": point + critical * standard_error,
        "n_years": g,
    }


def year_block_bootstrap_delta_rmse(
    y_true,
    pred_a,
    pred_b,
    years,
    n_boot: int = 2_000,
    random_state: int = RANDOM_STATE,
) -> dict:
    """Resample complete evaluation years; report RMSE(a) minus RMSE(b)."""
    yt, pa, pb = (np.asarray(v, dtype=float) for v in (y_true, pred_a, pred_b))
    yr = np.asarray(years)
    valid = np.isfinite(yt) & np.isfinite(pa) & np.isfinite(pb)
    yt, pa, pb, yr = yt[valid], pa[valid], pb[valid], yr[valid]
    unique_years = np.unique(yr)
    components = {}
    for year in unique_years:
        mask = yr == year
        components[year] = (
            float(np.sum((yt[mask] - pa[mask]) ** 2)),
            float(np.sum((yt[mask] - pb[mask]) ** 2)),
            int(mask.sum()),
        )

    def delta(selection) -> float:
        sse_a = sum(components[y][0] for y in selection)
        sse_b = sum(components[y][1] for y in selection)
        n = sum(components[y][2] for y in selection)
        return float(np.sqrt(sse_a / n) - np.sqrt(sse_b / n))

    if len(unique_years) == 0:
        return {"delta_RMSE": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n_years": 0}
    rng = np.random.default_rng(random_state)
    draws = np.array([
        delta(rng.choice(unique_years, size=len(unique_years), replace=True))
        for _ in range(n_boot)
    ])
    return {
        "delta_RMSE": delta(unique_years),
        "ci_low": float(np.percentile(draws, 2.5)),
        "ci_high": float(np.percentile(draws, 97.5)),
        "n_years": int(len(unique_years)),
    }

## 3. Load the Parquet panel efficiently

In [4]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Adjust DATA_PATH in Section 1."
    )

DROP_COLS_AT_LOAD = ["name", "dateinc", "report_date"]
schema_cols = pq.ParquetFile(DATA_PATH).schema_arrow.names
usecols = [c for c in schema_cols if c not in DROP_COLS_AT_LOAD]

df = pd.read_parquet(DATA_PATH, columns=usecols)

required = {ID_COL, YEAR_COL, TARGET}
missing_required = sorted(required.difference(df.columns))
if missing_required:
    raise KeyError(f"Required columns are missing: {missing_required}")

float64_cols = df.select_dtypes(include="float64").columns
df[float64_cols] = df[float64_cols].astype("float32")

df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce").astype("Int16")
df = df.dropna(subset=[ID_COL, YEAR_COL, TARGET]).copy()
df[YEAR_COL] = df[YEAR_COL].astype(int)

gc.collect()
print(f"Shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.head()

Shape: (1745282, 77)
Memory usage: 0.69 GB


,lm_positive,lm_negative,lm_polarity,uncertainty_ratio,litigious_ratio,constraining_ratio,strong_modal_ratio,weak_modal_ratio,ifo_business_climate,ifo_business_situation,ifo_business_expectations,de_economic_sentiment_index,de_employment_expectations_index,de_industry_confidence,de_services_confidence,de_consumer_confidence,de_retail_confidence,de_construction_confidence,year,ifo_business_climate_growth,ifo_business_situation_growth,ifo_business_expectations_growth,de_economic_sentiment_index_growth,de_employment_expectations_index_growth,de_industry_confidence_growth,de_services_confidence_growth,de_consumer_confidence_growth,de_retail_confidence_growth,de_construction_confidence_growth,idnr,type,naics_core_code,closdate_year,empl,ncliGrowthNextYear,fias,ifas,tfas,ofas,cuas,stok,debt,ocas,cash,toas,shfd,capi,osfd,ncli,ltdb,oncl,prov,culi,loan,cred,ocli,tshf,wkca,leverage,gearing,solvency,current_ratio,quick_ratio,cash_ratio,inventory_share,receivables_share,working_capital_ratio,log_toas,log_empl,ncliGrowthThisYear,toas_growth,cash_growth,growth_volatility,firm_age,years_in_panel,naics_2digit,is_na_empl
0,265,402,-0.205397,0.020557,0.014647,0.003769,0.001970,0.006167,98.199997,98.400002,98.099998,103.000000,102.099998,-4.4,12.300000,-1.4,-11.2,-11.4,2014,2.291667,2.393340,2.294056,0.684262,2.202202,-2.222222,7.894737,-26.315790,-15.151515,-10.236220,DE2010353555,Limited liability company - GmbH,4247,2014,60.0,0.384772,1116330.0,58274.0,1032556.0,25500.0,5473217.0,629038.0,3433723.0,1410457.0,642453.0,6589548.0,340819.0,300000.0,40819.0,530748.0,438352.0,92396.0,92396.0,5717980.0,279973.0,4845287.0,592720.0,6589548.0,-782527.0,0.080544,2.378744,0.051721,0.957194,0.847184,0.097496,0.095460,0.448893,-0.118054,15.700995,4.110874,NaN,NaN,NaN,NaN,24.0,0,42,0
1,301,447,-0.195187,0.018509,0.033679,0.014682,0.001600,0.004593,100.699997,100.599998,100.800003,106.400002,109.800003,-3.0,19.600000,-1.8,-2.5,-5.3,2015,-0.297030,0.099502,-0.787402,-0.467727,0.733945,15.384615,-8.837210,-45.454544,257.142853,-1.851852,DE2010353555,Limited liability company - GmbH,4247,2015,52.0,-0.401733,1040340.0,57800.0,952040.0,30500.0,5401788.0,540662.0,3050158.0,1810968.0,946953.0,6442129.0,634808.0,300000.0,334808.0,734965.0,492314.0,242651.0,242651.0,5072355.0,283886.0,4225698.0,562771.0,6442129.0,-634878.0,0.114087,1.604975,0.098540,1.064947,0.958357,0.146994,0.083926,0.448893,-0.098551,15.678370,3.970292,0.384772,-0.022372,0.473965,NaN,25.0,1,42,0
2,352,677,-0.315841,0.021990,0.012566,0.005463,0.002117,0.005600,101.000000,101.800003,100.199997,109.400002,112.099998,2.5,20.299999,-0.3,-4.0,5.5,2016,-0.590551,-0.391389,-0.792079,1.390176,0.358102,733.333313,7.407407,-66.666664,37.931034,150.000000,DE2010353555,Limited liability company - GmbH,4247,2016,49.0,0.816768,1286081.0,34945.0,1220636.0,30500.0,5791927.0,437567.0,3559991.0,1794370.0,964097.0,7078008.0,559668.0,300000.0,259668.0,439705.0,272022.0,167683.0,167683.0,6078635.0,220435.0,5323352.0,534848.0,7078008.0,-1325795.0,0.062123,1.179521,0.079071,0.952833,0.880849,0.136210,0.061821,0.448893,-0.118054,15.772503,3.912023,-0.401733,0.098706,0.018104,0.556143,26.0,2,42,0
3,397,547,-0.158898,0.016656,0.015763,0.006186,0.002141,0.004342,104.699997,107.099998,102.500000,114.900002,116.900002,14.5,21.500000,1.4,0.1,12.3,2017,-0.285714,0.469043,-0.870406,0.612960,2.453988,9.022556,3.365385,-6.666667,-104.545456,1.652893,DE2010353555,Limited liability company - GmbH,4247,2017,49.0,-0.247863,1151630.0,33402.0,1072728.0,45500.0,5822139.0,379907.0,3151085.0,2291146.0,1558041.0,6973769.0,488143.0,300000.0,188143.0,798842.0,648201.0,150641.0,150641.0,5686783.0,297649.0,4887944.0,501190.0,6973769.0,-1356952.0,0.114550,2.246250,0.069997,1.023802,0.956997,0.223414,0.054477,0.448893,-0.118054,15.757667,3.912023,0.816768,-0.014727,0.616062,0.617786,27.0,3,42,0
4,359,513,-0.176606,0.020424,0.012030,0.006295,0.002518,0.004337,101.199997,105.300003,97.300003,110.199997,115.699997,3.9,21.600000,-0.8,-5.0,18.9,2018,-1

## 4. Data integrity and panel descriptives

In [5]:
year_counts = df[YEAR_COL].value_counts().sort_index()
duplicate_firm_years = int(df.duplicated([ID_COL, YEAR_COL]).sum())

print("Observations by fiscal year:")
display(year_counts.to_frame("n"))
print(f"Duplicate firm-years: {duplicate_firm_years:,}")
print("Target summary:")
display(df[TARGET].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame().T)

# Accounting identity audit. These checks do not alter the data.
def relative_gap(a: pd.Series, b: pd.Series) -> pd.Series:
    denom = np.maximum(np.maximum(a.abs(), b.abs()), 1.0)
    return (a - b).abs() / denom

audit_rows = []
if {"toas", "tshf"}.issubset(df.columns):
    gap = relative_gap(df["toas"], df["tshf"])
    audit_rows.append({
        "identity": "toas = tshf",
        "median_relative_gap": float(gap.median()),
        "p99_relative_gap": float(gap.quantile(0.99)),
        "share_gap_gt_1pct": float((gap > 0.01).mean()),
    })
if {"fias", "cuas", "toas"}.issubset(df.columns):
    gap = relative_gap(df["fias"] + df["cuas"], df["toas"])
    audit_rows.append({
        "identity": "fias + cuas = toas",
        "median_relative_gap": float(gap.median()),
        "p99_relative_gap": float(gap.quantile(0.99)),
        "share_gap_gt_1pct": float((gap > 0.01).mean()),
    })
if {"shfd", "ncli", "culi", "tshf"}.issubset(df.columns):
    gap = relative_gap(df["shfd"] + df["ncli"] + df["culi"], df["tshf"])
    audit_rows.append({
        "identity": "shfd + ncli + culi = tshf",
        "median_relative_gap": float(gap.median()),
        "p99_relative_gap": float(gap.quantile(0.99)),
        "share_gap_gt_1pct": float((gap > 0.01).mean()),
    })

if audit_rows:
    display(pd.DataFrame(audit_rows))

firm_history = df.groupby(ID_COL)[YEAR_COL].agg(["min", "max", "count"])
print("Firm-history length:")
display(firm_history["count"].describe(percentiles=[0.5, 0.75, 0.9, 0.99]).to_frame().T)

Observations by fiscal year:


,n
closdate_year,
2010,13132
2011,12950
2012,21695
2013,39951
2014,138297
2015,145276
2016,149169
2017,190184
2018,201605


Duplicate firm-years: 0
Target summary:


,count,mean,std,min,1%,5%,50%,95%,99%,max
ncliGrowthNextYear,1745282.0,-0.056992,0.389922,-1.0,-0.97015,-0.799286,-0.041829,0.640229,0.907659,0.999999


,identity,median_relative_gap,p99_relative_gap,share_gap_gt_1pct
0,toas = tshf,0.0,0.000000,0.000000
1,fias + cuas = toas,0.0,0.000002,0.000158
2,shfd + ncli + culi = tshf,0.0,0.000004,0.000002


Firm-history length:


,count,mean,std,min,50%,75%,90%,99%,max
count,285318.0,6.116971,2.54548,1.0,7.0,8.0,9.0,9.0,9.0


The accounting audit is diagnostic. Large discrepancies may reflect missing components, differing statement definitions, or upstream unit problems. The model therefore prioritizes economically interpretable ratios and transformed levels. Extreme numeric values are handled by fold-fitted winsorization in the regularized-linear pipeline. The primary XGBoost models use the unscaled engineered features and native missing-value handling; a separate fold-fitted clipping robustness check evaluates whether this choice matters.


## 5. Leakage-safe feature engineering

In [6]:
def safe_ratio(
    numerator: pd.Series,
    denominator: pd.Series,
    *,
    require_positive_denominator: bool = True,
) -> pd.Series:
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    valid = numerator.notna() & denominator.notna()
    if require_positive_denominator:
        valid &= denominator > 0
    else:
        valid &= denominator != 0
    out = pd.Series(np.nan, index=numerator.index, dtype="float32")
    out.loc[valid] = (numerator.loc[valid] / denominator.loc[valid]).astype("float32")
    return out.replace([np.inf, -np.inf], np.nan)

# Sort once so all lagged firm features use only t and earlier observations.
df = df.sort_values([ID_COL, YEAR_COL]).copy()
firm_group = df.groupby(ID_COL, sort=False)
lag_year = firm_group[YEAR_COL].shift(1)
consecutive = (df[YEAR_COL] - lag_year).eq(1)

# -----------------------------------------------------------------------------
# Ratios based on documented balance-sheet identities. No full-sample
# clipping is performed here; fold-fitted winsorization is applied later.
# -----------------------------------------------------------------------------
if {"cuas", "culi"}.issubset(df.columns):
    df["current_ratio_clean"] = safe_ratio(df["cuas"], df["culi"])
    df["log1p_current_ratio_clean"] = np.log1p(
        df["current_ratio_clean"].clip(lower=0)
    ).astype("float32")

if {"cuas", "stok", "culi"}.issubset(df.columns):
    quick_assets = df["cuas"] - df["stok"]
    df["quick_ratio_clean"] = safe_ratio(quick_assets, df["culi"])
    df["log1p_quick_ratio_clean"] = np.log1p(
        df["quick_ratio_clean"].clip(lower=0)
    ).astype("float32")

if {"shfd", "toas"}.issubset(df.columns):
    df["solvency_ratio_clean"] = safe_ratio(df["shfd"], df["toas"])

if {"ncli", "loan", "shfd"}.issubset(df.columns):
    # Gearing is undefined or difficult to interpret with non-positive equity.
    gearing_num = df["ncli"] + df["loan"]
    df["gearing_ratio_clean"] = safe_ratio(gearing_num, df["shfd"])
    df["log1p_gearing_ratio_clean"] = np.log1p(
        df["gearing_ratio_clean"].clip(lower=0)
    ).astype("float32")

if {"ncli", "culi", "toas"}.issubset(df.columns):
    df["liabilities_to_assets"] = safe_ratio(df["ncli"] + df["culi"], df["toas"])

if {"wkca", "toas"}.issubset(df.columns):
    df["working_capital_to_assets"] = safe_ratio(
        df["wkca"], df["toas"], require_positive_denominator=True
    )

if {"stok", "cuas"}.issubset(df.columns):
    valid = (df["cuas"] > 0) & (df["stok"] >= 0) & (df["stok"] <= df["cuas"])
    df["inventory_share_current_assets"] = np.where(
        valid, df["stok"] / df["cuas"], np.nan
    ).astype("float32")

if {"debt", "cuas"}.issubset(df.columns):
    valid = (df["cuas"] > 0) & (df["debt"] >= 0) & (df["debt"] <= df["cuas"])
    df["receivables_share_current_assets"] = np.where(
        valid, df["debt"] / df["cuas"], np.nan
    ).astype("float32")

if {"cash", "toas"}.issubset(df.columns):
    valid = (df["toas"] > 0) & (df["cash"] >= 0) & (df["cash"] <= df["toas"])
    df["cash_to_assets"] = np.where(valid, df["cash"] / df["toas"], np.nan).astype("float32")

if {"fias", "toas"}.issubset(df.columns):
    valid = (df["toas"] > 0) & (df["fias"] >= 0) & (df["fias"] <= df["toas"])
    df["fixed_assets_share"] = np.where(valid, df["fias"] / df["toas"], np.nan).astype("float32")

if {"ltdb", "ncli"}.issubset(df.columns):
    valid = (df["ncli"] > 0) & (df["ltdb"] >= 0) & (df["ltdb"] <= df["ncli"])
    df["ltdb_share_of_ncli"] = np.where(valid, df["ltdb"] / df["ncli"], np.nan).astype("float32")

# -----------------------------------------------------------------------------
# Stable changes in firm levels. These replace unstable cash percentage growth
# and make the time direction explicit.
# -----------------------------------------------------------------------------
if "log_toas" not in df.columns and "toas" in df.columns:
    df["log_toas"] = np.log1p(df["toas"].clip(lower=0)).astype("float32")
if "log_empl" not in df.columns and "empl" in df.columns:
    df["log_empl"] = np.log1p(df["empl"].clip(lower=0)).astype("float32")

if "log_toas" in df.columns:
    lag_log_toas = firm_group["log_toas"].shift(1)
    df["log_toas_change_clean"] = np.where(
        consecutive, df["log_toas"] - lag_log_toas, np.nan
    ).astype("float32")

if {"cash", "toas"}.issubset(df.columns):
    lag_cash = firm_group["cash"].shift(1)
    lag_toas = firm_group["toas"].shift(1)
    valid = consecutive & lag_toas.gt(0)
    df["cash_change_to_lagged_assets"] = np.where(
        valid, (df["cash"] - lag_cash) / lag_toas, np.nan
    ).astype("float32")

if "ncliGrowthThisYear" in df.columns:
    rolling_volatility = (
        df.groupby(ID_COL, sort=False)["ncliGrowthThisYear"]
          .rolling(window=3, min_periods=2)
          .std()
          .reset_index(level=0, drop=True)
    )
    df["ncli_growth_volatility_3y"] = rolling_volatility.astype("float32")

# -----------------------------------------------------------------------------
# Macro transformations.
# Confidence balances can cross zero, so use first differences.
# Positive indices use percentage changes calculated from annual levels.
# -----------------------------------------------------------------------------
BALANCE_LEVELS = [
    "de_construction_confidence",
    "de_consumer_confidence",
    "de_industry_confidence",
    "de_retail_confidence",
    "de_services_confidence",
]
POSITIVE_INDEX_LEVELS = [
    "de_economic_sentiment_index",
    "de_employment_expectations_index",
    "ifo_business_climate",
    "ifo_business_expectations",
    "ifo_business_situation",
]

available_macro_levels = [
    c for c in BALANCE_LEVELS + POSITIVE_INDEX_LEVELS if c in df.columns
]
if available_macro_levels:
    annual_macro = (
        df.groupby(YEAR_COL, sort=True)[available_macro_levels]
          .median(numeric_only=True)
          .sort_index()
    )

    for col in [c for c in BALANCE_LEVELS if c in annual_macro.columns]:
        new_col = f"{col}_change_clean"
        year_values = annual_macro[col].diff()
        df[new_col] = df[YEAR_COL].map(year_values).astype("float32")

    for col in [c for c in POSITIVE_INDEX_LEVELS if c in annual_macro.columns]:
        new_col = f"{col}_pct_change_clean"
        current = annual_macro[col]
        previous = current.shift(1)
        year_values = ((current / previous) - 1).where((current > 0) & (previous > 0))
        df[new_col] = df[YEAR_COL].map(year_values).astype("float32")

# -----------------------------------------------------------------------------
# Text counts are used only if a document-length variable exists. Otherwise,
# raw lm_positive/lm_negative counts are omitted and polarity/ratios are used.
# -----------------------------------------------------------------------------
TOKEN_COUNT_CANDIDATES = [
    "token_count", "n_tokens", "word_count", "total_words", "document_length"
]
token_count_col = next((c for c in TOKEN_COUNT_CANDIDATES if c in df.columns), None)
if token_count_col is not None:
    for source, target in [
        ("lm_positive", "lm_positive_frequency"),
        ("lm_negative", "lm_negative_frequency"),
    ]:
        if source in df.columns:
            df[target] = safe_ratio(df[source], df[token_count_col])
    print(f"Normalized sentiment counts using: {token_count_col}")
else:
    print("No document-length column found: raw positive/negative counts will be omitted.")

# Replace any remaining infinities and downcast new floating-point columns.
float64_cols = df.select_dtypes(include="float64").columns
df[float64_cols] = df[float64_cols].astype("float32")
df.replace([np.inf, -np.inf], np.nan, inplace=True)

del firm_group, lag_year, consecutive
gc.collect()

No document-length column found: raw positive/negative counts will be omitted.


0

### 5.1 Feature-engineering audit

In [7]:
engineered_candidates = [
    "current_ratio_clean", "log1p_current_ratio_clean",
    "quick_ratio_clean", "log1p_quick_ratio_clean",
    "solvency_ratio_clean", "gearing_ratio_clean",
    "log1p_gearing_ratio_clean", "liabilities_to_assets",
    "working_capital_to_assets", "inventory_share_current_assets",
    "receivables_share_current_assets", "cash_to_assets",
    "fixed_assets_share", "ltdb_share_of_ncli",
    "log_toas_change_clean", "cash_change_to_lagged_assets",
    "ncli_growth_volatility_3y",
]
engineered_candidates += [
    f"{c}_change_clean" for c in BALANCE_LEVELS
] + [
    f"{c}_pct_change_clean" for c in POSITIVE_INDEX_LEVELS
]
engineered_candidates += ["lm_positive_frequency", "lm_negative_frequency"]
engineered_present = [c for c in engineered_candidates if c in df.columns]

feature_audit = pd.DataFrame({
    "missing_share": df[engineered_present].isna().mean(),
    "n_unique": df[engineered_present].nunique(dropna=True),
    "min": df[engineered_present].min(numeric_only=True),
    "median": df[engineered_present].median(numeric_only=True),
    "max": df[engineered_present].max(numeric_only=True),
}).sort_values("missing_share")
display(feature_audit)

,missing_share,n_unique,min,median,max
liabilities_to_assets,0.000000,1666960,-0.124217,0.649108,7.342030e+05
solvency_ratio_clean,0.000000,1698084,-734202.000000,0.350892,1.124218e+00
fixed_assets_share,0.000018,1674087,0.000000,0.237142,1.000000e+00
ltdb_share_of_ncli,0.000128,1192410,0.000000,0.709300,1.000000e+00
de_retail_confidence_change_clean,0.007524,13,-18.400000,-1.000000,9.500000e+00
de_industry_confidence_change_clean,0.007524,13,-20.299999,1.400000,2.440000e+01
ifo_business_expectations_pct_change_clean,0.007524,13,-0.096591,-0.013727,6.827736e-02
ifo_business_situation_pct_change_clean,0.007524,13,-0.073663,-0.007064,6.100214e-02
ifo_business_climate_pct_change_clean,0.007524,13,-0.063941,-0.021912,5.241096e-02
de_employment_expectations_index_pct_change_clean,0.007524,13,-0.093345,-0.010265,1.725206e-01


## 6. Explicit feature blocks

In [8]:
def present(columns: list[str]) -> list[str]:
    return [c for c in columns if c in df.columns]

CATEGORY_COLS = present(["naics_2digit", "type"])

FIRM_NUMERIC_CANDIDATES = [
    "log_toas",
    "log_empl",                 # raw empl is intentionally omitted
    "firm_age",
    "years_in_panel",
    "is_na_empl",
    "ncliGrowthThisYear",
    "log_toas_change_clean",
    "cash_change_to_lagged_assets",
    "ncli_growth_volatility_3y",
    "log1p_current_ratio_clean",
    "log1p_quick_ratio_clean",
    "solvency_ratio_clean",
    "log1p_gearing_ratio_clean",
    "liabilities_to_assets",
    "working_capital_to_assets",
    "inventory_share_current_assets",
    "receivables_share_current_assets",
    "cash_to_assets",
    "fixed_assets_share",
    "ltdb_share_of_ncli",
]
FIRM_FEATURES = present(FIRM_NUMERIC_CANDIDATES) + CATEGORY_COLS

SURVEY_LEVEL_FEATURES = present(BALANCE_LEVELS + POSITIVE_INDEX_LEVELS)
SURVEY_CHANGE_FEATURES = present(
    [f"{c}_change_clean" for c in BALANCE_LEVELS]
    + [f"{c}_pct_change_clean" for c in POSITIVE_INDEX_LEVELS]
)
SURVEY_FEATURES = SURVEY_LEVEL_FEATURES + SURVEY_CHANGE_FEATURES

TEXT_FEATURES = present([
    "lm_polarity",
    "uncertainty_ratio",
    "litigious_ratio",
    "constraining_ratio",
    "strong_modal_ratio",
    "weak_modal_ratio",
    "lm_positive_frequency",
    "lm_negative_frequency",
])

FEATURE_SETS = {
    "firm": FIRM_FEATURES,
    "firm_survey": list(dict.fromkeys(FIRM_FEATURES + SURVEY_FEATURES)),
    "firm_survey_text": list(dict.fromkeys(FIRM_FEATURES + SURVEY_FEATURES + TEXT_FEATURES)),
}

for name, cols in FEATURE_SETS.items():
    if not cols:
        raise ValueError(f"Feature set {name!r} is empty.")
    print(f"{name:>18}: {len(cols):>3} columns")
    print(cols)

# Explicit leakage / redundancy guard.
FORBIDDEN_FEATURES = {
    TARGET, YEAR_COL, "year", ID_COL, "naics_core_code", "empl",
    "cash_growth", "toas_growth",
    "de_construction_confidence_growth", "de_consumer_confidence_growth",
    "de_industry_confidence_growth", "de_retail_confidence_growth",
    "de_services_confidence_growth",
    "lm_positive", "lm_negative",
}
for set_name, cols in FEATURE_SETS.items():
    overlap = sorted(set(cols).intersection(FORBIDDEN_FEATURES))
    if overlap:
        raise AssertionError(f"Forbidden features in {set_name}: {overlap}")

ABLATION_SETS = ["firm", "firm_survey", "firm_survey_text"]

              firm:  22 columns
['log_toas', 'log_empl', 'firm_age', 'years_in_panel', 'is_na_empl', 'ncliGrowthThisYear', 'log_toas_change_clean', 'cash_change_to_lagged_assets', 'ncli_growth_volatility_3y', 'log1p_current_ratio_clean', 'log1p_quick_ratio_clean', 'solvency_ratio_clean', 'log1p_gearing_ratio_clean', 'liabilities_to_assets', 'working_capital_to_assets', 'inventory_share_current_assets', 'receivables_share_current_assets', 'cash_to_assets', 'fixed_assets_share', 'ltdb_share_of_ncli', 'naics_2digit', 'type']
       firm_survey:  42 columns
['log_toas', 'log_empl', 'firm_age', 'years_in_panel', 'is_na_empl', 'ncliGrowthThisYear', 'log_toas_change_clean', 'cash_change_to_lagged_assets', 'ncli_growth_volatility_3y', 'log1p_current_ratio_clean', 'log1p_quick_ratio_clean', 'solvency_ratio_clean', 'log1p_gearing_ratio_clean', 'liabilities_to_assets', 'working_capital_to_assets', 'inventory_share_current_assets', 'receivables_share_current_assets', 'cash_to_assets', 'fixed_asset

The nested feature blocks answer the substantive question directly:

1. `firm`: accounting and firm characteristics;
2. `firm_survey`: firm variables plus macro/survey information;
3. `firm_survey_text`: the preceding block plus text indicators.

Hyperparameters are selected separately for each block, so a richer block is not disadvantaged by settings tuned for a different information set.

## 7. Development, tuning, and frozen test samples

In [9]:
ALL_MODEL_FEATURES = FEATURE_SETS["firm_survey_text"]
df[ALL_MODEL_FEATURES] = df[ALL_MODEL_FEATURES].replace([np.inf, -np.inf], np.nan)

year_counts = df[YEAR_COL].value_counts().sort_index()
THIN_YEARS = sorted(int(y) for y, n in year_counts.items() if n < MIN_YEAR_OBSERVATIONS)
HEADLINE_TEST_YEARS = sorted(
    int(y) for y in year_counts.index
    if y >= TEST_START_YEAR and y not in THIN_YEARS
)
THIN_TEST_YEARS = sorted(
    int(y) for y in year_counts.index
    if y >= TEST_START_YEAR and y in THIN_YEARS
)

train_df = df[df[YEAR_COL] < TEST_START_YEAR].copy()
test_df = df[df[YEAR_COL].isin(HEADLINE_TEST_YEARS)].copy()
thin_test_df = df[df[YEAR_COL].isin(THIN_TEST_YEARS)].copy()

if train_df.empty or test_df.empty:
    raise ValueError("Development or headline test set is empty. Check year thresholds.")

train_mean = float(train_df[TARGET].mean())
print(f"Development: {train_df.shape}, years {train_df[YEAR_COL].min()}-{train_df[YEAR_COL].max()}")
print(f"Headline held-out test: {test_df.shape}, years {HEADLINE_TEST_YEARS}")
print(f"Thin held-out years, evaluated separately: {THIN_TEST_YEARS}")
print(f"Development mean outcome: {train_mean:.6f}")

train_firms = set(train_df[ID_COL].unique())
test_firms = set(test_df[ID_COL].unique())
print(f"Firm overlap, development to headline test: {len(train_firms & test_firms):,} firms "
      f"({len(train_firms & test_firms) / max(len(test_firms), 1):.1%} of test firms)")

# Deterministic firm-level samples for the two-stage search.
screen_df = sample_firms_to_target(train_df, SCREEN_ROWS_PER_YEAR)
tuning_df = sample_firms_to_target(train_df, TUNING_ROWS_PER_YEAR)

for name, frame in [("screen_df", screen_df), ("tuning_df", tuning_df)]:
    print(f"\n{name}: {len(frame):,} rows, {frame[ID_COL].nunique():,} firms")
    display(frame[YEAR_COL].value_counts().sort_index().to_frame("n").T)

screen_splits, screen_years = expanding_window_folds(
    screen_df, FIRST_VALIDATION_YEAR, LAST_VALIDATION_YEAR
)
cv_splits, cv_years = expanding_window_folds(
    tuning_df, FIRST_VALIDATION_YEAR, LAST_VALIDATION_YEAR
)

fold_summary = pd.DataFrame([{
    "fold": i,
    "train_start": int(tuning_df.iloc[train_idx][YEAR_COL].min()),
    "train_end": int(tuning_df.iloc[train_idx][YEAR_COL].max()),
    "validation_year": year,
    "n_train": len(train_idx),
    "n_validation": len(validation_idx),
} for i, ((train_idx, validation_idx), year) in enumerate(zip(cv_splits, cv_years), start=1)])
display(fold_summary)
print("Look-ahead check passed: every training window ends before its validation year.")

Development: (1113369, 104), years 2010-2019
Headline held-out test: (630228, 104), years [2020, 2021, 2022]
Thin held-out years, evaluated separately: [2023]
Development mean outcome: -0.068890
Firm overlap, development to headline test: 238,249 firms (86.7% of test firms)

screen_df: 110,894 rows, 24,755 firms


closdate_year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
n,1348,1293,2128,3943,13688,14461,14891,18927,20162,20053



tuning_df: 276,918 rows, 61,752 firms


closdate_year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
n,3263,3252,5398,9867,34334,36060,37119,47325,50255,50045


,fold,train_start,train_end,validation_year,n_train,n_validation
0,1,2010,2014,2015,56114,36060
1,2,2010,2015,2016,92174,37119
2,3,2010,2016,2017,129293,47325
3,4,2010,2017,2018,176618,50255
4,5,2010,2018,2019,226873,50045


Look-ahead check passed: every training window ends before its validation year.


## 8. Benchmarks, preprocessing, and model factories

In [10]:
def baseline_predictions(frame: pd.DataFrame, train_mean_value: float) -> dict[str, np.ndarray]:
    predictions = {
        "Mean": np.full(len(frame), train_mean_value, dtype=float),
        "Zero growth": np.zeros(len(frame), dtype=float),
    }
    if "ncliGrowthThisYear" in frame.columns:
        persistence = pd.to_numeric(frame["ncliGrowthThisYear"], errors="coerce").to_numpy(float)
        persistence = np.where(np.isfinite(persistence), persistence, train_mean_value)
        predictions["Persistence"] = persistence
    return predictions


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:  # scikit-learn < 1.2
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def make_preprocessor(feature_cols: list[str]) -> ColumnTransformer:
    """Linear preprocessing; every transformation is estimated inside the fold."""
    categorical = [c for c in CATEGORY_COLS if c in feature_cols]
    numeric = [c for c in feature_cols if c not in categorical]

    numeric_pipe = Pipeline([
        ("winsor", QuantileWinsorizer(WINSOR_LOWER, WINSOR_UPPER)),
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", make_one_hot_encoder()),
        # with_mean=False preserves sparsity while placing dummies on comparable penalty scales.
        ("scale", StandardScaler(with_mean=False)),
    ])

    transformers = []
    if numeric:
        transformers.append(("num", numeric_pipe, numeric))
    if categorical:
        transformers.append(("cat", categorical_pipe, categorical))
    # Force a sparse combined design to reduce peak memory in the full-data refit.
    return ColumnTransformer(transformers, sparse_threshold=1.0)


def make_linear_estimator(family: str, parameters: dict | None = None):
    parameters = parameters or {}
    if family == "Ridge":
        return Ridge(solver="lsqr", max_iter=20_000, **parameters)
    if family == "Lasso":
        return Lasso(
            max_iter=LINEAR_MAX_ITER,
            tol=1e-4,
            selection="random",
            random_state=RANDOM_STATE,
            **parameters,
        )
    if family == "ElasticNet":
        return ElasticNet(
            max_iter=LINEAR_MAX_ITER,
            tol=1e-4,
            selection="random",
            random_state=RANDOM_STATE,
            **parameters,
        )
    raise ValueError(f"Unknown linear family: {family}")


def make_linear_pipeline(family: str, parameters: dict | None, feature_cols: list[str]) -> Pipeline:
    return Pipeline([
        ("preprocess", make_preprocessor(feature_cols)),
        ("model", make_linear_estimator(family, parameters)),
    ])


def tree_category_levels(train_frame: pd.DataFrame, feature_cols: list[str]) -> dict[str, pd.Index]:
    return {
        c: pd.Index(sorted(train_frame[c].dropna().astype(str).unique()))
        for c in CATEGORY_COLS if c in feature_cols
    }


def prepare_xgb_frame(
    frame: pd.DataFrame,
    feature_cols: list[str],
    category_levels: dict[str, pd.Index] | None = None,
) -> pd.DataFrame:
    out = frame[feature_cols].copy()
    for col in [c for c in CATEGORY_COLS if c in feature_cols]:
        values = out[col].astype("string")
        if category_levels is None:
            out[col] = values.astype("category")
        else:
            allowed = category_levels[col]
            values = values.where(values.isin(allowed))
            out[col] = pd.Categorical(values, categories=allowed)
    return out


def make_xgb_estimator(parameters: dict) -> xgb.XGBRegressor:
    return xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        enable_categorical=True,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        **parameters,
    )


def fit_predict_model(
    model_name: str,
    parameters: dict,
    feature_cols: list[str],
    train_frame: pd.DataFrame,
    score_frame: pd.DataFrame,
):
    if model_name in {"Ridge", "Lasso", "ElasticNet"}:
        model = make_linear_pipeline(model_name, parameters, feature_cols)
        model.fit(train_frame[feature_cols], train_frame[TARGET])
        prediction = model.predict(score_frame[feature_cols])
        return np.asarray(prediction, dtype=float), {"model": model, "category_levels": None}

    if model_name == "XGBoost":
        levels = tree_category_levels(train_frame, feature_cols)
        X_train = prepare_xgb_frame(train_frame, feature_cols, levels)
        X_score = prepare_xgb_frame(score_frame, feature_cols, levels)
        model = make_xgb_estimator(parameters)
        model.fit(X_train, train_frame[TARGET], verbose=False)
        prediction = model.predict(X_score)
        del X_train, X_score
        return np.asarray(prediction, dtype=float), {"model": model, "category_levels": levels}

    raise ValueError(f"Unknown model: {model_name}")


def predict_fitted_artifact(artifact: dict, model_name: str, feature_cols: list[str], frame: pd.DataFrame):
    model = artifact["model"]
    if model_name in {"Ridge", "Lasso", "ElasticNet"}:
        return np.asarray(model.predict(frame[feature_cols]), dtype=float)
    X = prepare_xgb_frame(frame, feature_cols, artifact["category_levels"])
    prediction = np.asarray(model.predict(X), dtype=float)
    del X
    return prediction

## 9. Time-aware tuning of regularized linear models

In [11]:
if QUICK_MODE:
    LINEAR_GRIDS = {
        "Ridge": {"model__alpha": np.logspace(0, 6, 7)},
        "Lasso": {"model__alpha": np.logspace(-4, -1, 4)},
        "ElasticNet": {
            "model__alpha": np.logspace(-4, -1, 4),
            "model__l1_ratio": [0.1, 0.5, 0.9],
        },
    }
else:
    LINEAR_GRIDS = {
        "Ridge": {"model__alpha": np.logspace(-2, 7, 19)},
        "Lasso": {"model__alpha": np.logspace(-4, 0, 9)},
        "ElasticNet": {
            "model__alpha": np.logspace(-4, 0, 9),
            "model__l1_ratio": [0.1, 0.5, 0.9],
        },
    }

full_feature_cols = FEATURE_SETS["firm_survey_text"]

# Diagnostic only: survey variables contain far fewer independent annual realizations
# than the number of firm-year rows suggests.
survey_present = [c for c in SURVEY_FEATURES if c in full_feature_cols]
if survey_present:
    annual_survey = (
        tuning_df.groupby(YEAR_COL)[survey_present]
        .median(numeric_only=True)
        .dropna(axis=1, how="all")
        .dropna(axis=0)
    )
    standardized = (
        (annual_survey - annual_survey.mean())
        / annual_survey.std().replace(0, np.nan)
    ).dropna(axis=1)
    if standardized.shape[0] >= 2 and standardized.shape[1] >= 1:
        singular_values = np.linalg.svd(standardized.to_numpy(), compute_uv=False)
        condition_number = singular_values[0] / max(singular_values[-1], 1e-12)
        explained = singular_values ** 2 / np.sum(singular_values ** 2)
        print(f"Survey block: {standardized.shape[1]} series across {standardized.shape[0]} complete years")
        print(f"Condition number: {condition_number:,.0f}")
        print(f"Variance explained by first 3 components: {explained[:3].sum():.1%}")

family_rows = []
family_parameters = {}
linear_boundary_warnings = []

for family, grid in LINEAR_GRIDS.items():
    started = time.time()
    print(f"Tuning {family} on the full information block...")
    search = GridSearchCV(
        estimator=make_linear_pipeline(family, None, full_feature_cols),
        param_grid=grid,
        scoring="neg_root_mean_squared_error",
        cv=cv_splits,
        refit=False,
        n_jobs=LINEAR_SEARCH_N_JOBS,
        return_train_score=False,
        error_score="raise",
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        search.fit(tuning_df[full_feature_cols], tuning_df[TARGET])

    selected, diagnostics = select_one_standard_error(
        search.cv_results_, "model__alpha", len(cv_splits)
    )
    clean = {key.replace("model__", ""): value for key, value in selected.items()}
    family_parameters[family] = clean
    linear_boundary_warnings += check_grid_boundaries(selected, grid, family)
    family_rows.append({
        "family": family,
        "minimum_CV_RMSE": diagnostics["minimum_RMSE"],
        "one_SE_CV_RMSE": diagnostics["chosen_RMSE"],
        "one_SE": diagnostics["one_se"],
        "n_within_one_SE": diagnostics["n_within_one_se"],
        "minimum_parameters": json.dumps(diagnostics["minimum_parameters"], default=float),
        "selected_parameters": json.dumps(clean, default=float),
        "seconds": round(time.time() - started, 1),
    })
    del search
    gc.collect()

linear_family_results = pd.DataFrame(family_rows).sort_values("one_SE_CV_RMSE")
display(linear_family_results)
BEST_LINEAR_FAMILY = str(linear_family_results.iloc[0]["family"])
print("Selected linear family:", BEST_LINEAR_FAMILY)

# Retune the selected family separately for each information block.
LINEAR_BLOCK_PARAMS: dict[str, dict] = {}
linear_block_rows = []
convergence_rows = []

for block_name in ABLATION_SETS:
    feature_cols = FEATURE_SETS[block_name]
    grid = LINEAR_GRIDS[BEST_LINEAR_FAMILY]
    search = GridSearchCV(
        estimator=make_linear_pipeline(BEST_LINEAR_FAMILY, None, feature_cols),
        param_grid=grid,
        scoring="neg_root_mean_squared_error",
        cv=cv_splits,
        refit=False,
        n_jobs=LINEAR_SEARCH_N_JOBS,
        return_train_score=False,
        error_score="raise",
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        search.fit(tuning_df[feature_cols], tuning_df[TARGET])

    selected, diagnostics = select_one_standard_error(
        search.cv_results_, "model__alpha", len(cv_splits)
    )
    parameters = {key.replace("model__", ""): value for key, value in selected.items()}
    LINEAR_BLOCK_PARAMS[block_name] = parameters
    linear_boundary_warnings += check_grid_boundaries(
        selected, grid, f"{BEST_LINEAR_FAMILY}/{block_name}"
    )
    linear_block_rows.append({
        "feature_set": block_name,
        "family": BEST_LINEAR_FAMILY,
        "minimum_CV_RMSE": diagnostics["minimum_RMSE"],
        "one_SE_CV_RMSE": diagnostics["chosen_RMSE"],
        "parameters": json.dumps(parameters, default=float),
    })

    audit_model = make_linear_pipeline(BEST_LINEAR_FAMILY, parameters, feature_cols)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", category=ConvergenceWarning)
        audit_model.fit(tuning_df[feature_cols], tuning_df[TARGET])
    estimator = audit_model.named_steps["model"]
    if BEST_LINEAR_FAMILY == "Ridge":
        n_iter, converged = np.nan, True
    else:
        n_iter = int(np.max(np.atleast_1d(estimator.n_iter_)))
        converged = bool(n_iter < estimator.max_iter)
    convergence_rows.append({
        "feature_set": block_name,
        "family": BEST_LINEAR_FAMILY,
        "n_iter": n_iter,
        "max_iter": getattr(estimator, "max_iter", np.nan),
        "converged": converged,
        "convergence_warnings": sum(
            issubclass(w.category, ConvergenceWarning) for w in caught
        ),
        "nonzero_coefficients": int(np.sum(estimator.coef_ != 0)),
        "total_coefficients": int(estimator.coef_.size),
    })
    del search, audit_model
    gc.collect()

linear_block_results = pd.DataFrame(linear_block_rows)
convergence_audit = pd.DataFrame(convergence_rows)
display(linear_block_results)
display(convergence_audit)

if not convergence_audit["converged"].all():
    raise RuntimeError(
        "At least one selected coordinate-descent model did not converge. "
        "Do not continue to the held-out test; increase the minimum alpha or use Ridge."
    )

if linear_boundary_warnings:
    print("GRID BOUNDARY WARNINGS:")
    for message in sorted(set(linear_boundary_warnings)):
        print("  " + message)
else:
    print("No selected linear hyperparameter is at a grid boundary.")

linear_family_results.to_csv(OUTPUT_DIR / "linear_family_tuning.csv", index=False)
linear_block_results.to_csv(OUTPUT_DIR / "linear_block_tuning.csv", index=False)
convergence_audit.to_csv(OUTPUT_DIR / "linear_convergence_audit.csv", index=False)

Survey block: 20 series across 9 complete years
Condition number: 1,007,010
Variance explained by first 3 components: 91.6%
Tuning Ridge on the full information block...
Tuning Lasso on the full information block...
Tuning ElasticNet on the full information block...


,family,minimum_CV_RMSE,one_SE_CV_RMSE,one_SE,n_within_one_SE,minimum_parameters,selected_parameters,seconds
1,Lasso,0.384109,0.386661,0.003871,6,"{""model__alpha"": 0.0001}","{""alpha"": 0.03162277660168379}",198.8
2,ElasticNet,0.384063,0.387418,0.003846,20,"{""model__alpha"": 0.0001, ""model__l1_ratio"": 0.5}","{""alpha"": 0.31622776601683794, ""l1_ratio"": 0.1}",1529.6
0,Ridge,0.384447,0.387620,0.004439,16,"{""model__alpha"": 3162.2776601683795}","{""alpha"": 316227.7660168379}",136.8


Selected linear family: Lasso


,feature_set,family,minimum_CV_RMSE,one_SE_CV_RMSE,parameters
0,firm,Lasso,0.382169,0.383206,"{""alpha"": 0.01}"
1,firm_survey,Lasso,0.383940,0.386679,"{""alpha"": 0.03162277660168379}"
2,firm_survey_text,Lasso,0.384109,0.386661,"{""alpha"": 0.03162277660168379}"


,feature_set,family,n_iter,max_iter,converged,convergence_warnings,nonzero_coefficients,total_coefficients
0,firm,Lasso,19,10000,True,0,7,81
1,firm_survey,Lasso,18,10000,True,0,2,111
2,firm_survey_text,Lasso,22,10000,True,0,3,117


GRID BOUNDARY WARNINGS:
  ElasticNet: model__l1_ratio=0.1 is at the LOWER grid boundary.


### 9.1 Optional robustness: compare Ridge, Lasso, and Elastic Net within every information block

The primary workflow selects the regularization family on the complete information block and then
retunes that family within each nested block. The robustness check below removes that asymmetry:
each family is tuned separately within each block using the same expanding-window folds and the
same one-standard-error rule.

This table is diagnostic. It does **not** overwrite `BEST_LINEAR_FAMILY`, the frozen specification,
or any held-out result.


In [ ]:
linear_family_by_block_results = pd.DataFrame()

if RUN_LINEAR_FAMILY_BY_BLOCK_ROBUSTNESS:
    robustness_rows = []

    # Reuse the already-computed full-block family comparison.
    for _, row in linear_family_results.iterrows():
        robustness_rows.append({
            "feature_set": "firm_survey_text",
            "family": str(row["family"]),
            "minimum_CV_RMSE": float(row["minimum_CV_RMSE"]),
            "one_SE_CV_RMSE": float(row["one_SE_CV_RMSE"]),
            "selected_parameters": row["selected_parameters"],
            "source": "reused from Section 9",
        })

    # The remaining two blocks require additional development-only searches.
    for block_name in ["firm", "firm_survey"]:
        feature_cols = FEATURE_SETS[block_name]
        for family, grid in LINEAR_GRIDS.items():
            started = time.time()
            print(f"[robustness] tuning {family} on {block_name}...")
            search = GridSearchCV(
                estimator=make_linear_pipeline(family, None, feature_cols),
                param_grid=grid,
                scoring="neg_root_mean_squared_error",
                cv=cv_splits,
                refit=False,
                n_jobs=LINEAR_SEARCH_N_JOBS,
                return_train_score=False,
                error_score="raise",
            )
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=ConvergenceWarning)
                search.fit(tuning_df[feature_cols], tuning_df[TARGET])

            selected, diagnostics = select_one_standard_error(
                search.cv_results_, "model__alpha", len(cv_splits)
            )
            parameters = {
                key.replace("model__", ""): value for key, value in selected.items()
            }
            robustness_rows.append({
                "feature_set": block_name,
                "family": family,
                "minimum_CV_RMSE": diagnostics["minimum_RMSE"],
                "one_SE_CV_RMSE": diagnostics["chosen_RMSE"],
                "selected_parameters": json.dumps(parameters, default=float, sort_keys=True),
                "source": "additional block-specific search",
                "seconds": round(time.time() - started, 1),
            })
            del search
            gc.collect()

    linear_family_by_block_results = (
        pd.DataFrame(robustness_rows)
        .sort_values(["feature_set", "one_SE_CV_RMSE", "family"])
        .reset_index(drop=True)
    )
    display(linear_family_by_block_results.round(6))

    winners = (
        linear_family_by_block_results
        .sort_values(["feature_set", "one_SE_CV_RMSE"])
        .groupby("feature_set", as_index=False)
        .first()[["feature_set", "family", "one_SE_CV_RMSE", "selected_parameters"]]
    )
    print("Best regularized-linear family within each information block:")
    display(winners.round(6))

    if winners["family"].nunique() > 1:
        print(
            "The preferred penalty varies across information blocks. Treat the main "
            "single-family comparison as a disciplined simplification and report this robustness table."
        )
    else:
        print(
            "The same penalty family wins in every information block, supporting the primary "
            "single-family simplification."
        )

    linear_family_by_block_results.to_csv(
        OUTPUT_DIR / "linear_family_by_block_robustness.csv", index=False
    )
else:
    print(
        "RUN_LINEAR_FAMILY_BY_BLOCK_ROBUSTNESS=False: skipping the additional "
        "development-only linear searches."
    )


[robustness] tuning Ridge on firm...
[robustness] tuning Lasso on firm...
[robustness] tuning ElasticNet on firm...
[robustness] tuning Ridge on firm_survey...


The one-standard-error rule avoids treating tiny fourth-decimal CV differences as strong evidence for an almost unregularized model. The convergence audit is a hard gate: an unconverged Lasso or Elastic Net is not allowed to proceed to interpretation or final testing.

## 10. Time-aware XGBoost tuning without validation-fold early stopping

In [ ]:
if QUICK_MODE:
    XGB_GRID = {
        "n_estimators": [200, 400, 800],
        "max_depth": [2, 3, 4, 6],
        "learning_rate": [0.01, 0.03, 0.05, 0.10],
        "min_child_weight": [1, 20, 100, 300],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.4, 0.6, 0.8, 1.0],
        "reg_lambda": [0.1, 1.0, 20.0, 100.0, 500.0],
        "reg_alpha": [0.0, 0.1, 1.0],
    }
else:
    XGB_GRID = {
        "n_estimators": [200, 400, 800, 1_600, 3_000],
        "max_depth": [2, 3, 4, 6, 8, 10],
        "learning_rate": [0.005, 0.01, 0.03, 0.05, 0.10, 0.20],
        "min_child_weight": [1, 10, 50, 200, 1_000, 5_000],
        "subsample": [0.5, 0.7, 0.9, 1.0],
        "colsample_bytree": [0.3, 0.5, 0.7, 1.0],
        "reg_lambda": [0.1, 1.0, 5.0, 20.0, 100.0, 500.0, 2_000.0],
        "reg_alpha": [0.0, 0.01, 0.1, 1.0, 10.0, 50.0],
    }


def score_xgb_candidate(
    frame: pd.DataFrame,
    splits: list[tuple[np.ndarray, np.ndarray]],
    validation_years: list[int],
    feature_cols: list[str],
    parameters: dict,
) -> dict[str, float]:
    actual_parts = []
    prediction_parts = []
    benchmark_parts = []
    year_parts = []

    for (train_idx, validation_idx), validation_year in zip(splits, validation_years):
        fold_train = frame.iloc[train_idx]
        fold_validation = frame.iloc[validation_idx]
        fold_mean = float(fold_train[TARGET].mean())
        prediction, artifact = fit_predict_model(
            "XGBoost", parameters, feature_cols, fold_train, fold_validation
        )
        actual_parts.append(fold_validation[TARGET].to_numpy(dtype=float))
        prediction_parts.append(prediction)
        benchmark_parts.append(np.full(len(fold_validation), fold_mean, dtype=float))
        year_parts.append(np.full(len(fold_validation), validation_year, dtype=int))
        del artifact, prediction
        gc.collect()

    actual = np.concatenate(actual_parts)
    prediction = np.concatenate(prediction_parts)
    benchmark = np.concatenate(benchmark_parts)
    years = np.concatenate(year_parts)
    metrics = regression_metrics(actual, prediction, benchmark, years=years)
    per_year_rmse = [
        regression_metrics(actual[years == year], prediction[years == year], benchmark[years == year])["RMSE"]
        for year in np.unique(years)
    ]
    return {
        **metrics,
        "mean_year_RMSE": float(np.mean(per_year_rmse)),
        "worst_year_RMSE": float(np.max(per_year_rmse)),
    }


def tune_xgb_block(block_name: str, feature_cols: list[str]):
    candidates = list(ParameterSampler(
        XGB_GRID,
        n_iter=XGB_SCREEN_TRIALS,
        random_state=RANDOM_STATE,
    ))
    screen_rows = []
    started = time.time()
    for candidate_no, candidate in enumerate(candidates, start=1):
        metrics = score_xgb_candidate(
            screen_df, screen_splits, screen_years, feature_cols, candidate
        )
        screen_rows.append({
            "candidate": candidate_no,
            **metrics,
            "parameters": json.dumps(candidate, default=float, sort_keys=True),
        })
        print(
            f"[{block_name}] screen {candidate_no}/{len(candidates)} "
            f"RMSE={metrics['RMSE']:.6f}",
            end="\r",
        )
    screen_results = pd.DataFrame(screen_rows).sort_values("RMSE").reset_index(drop=True)
    print(f"[{block_name}] screening completed in {time.time() - started:.0f}s")

    refine_rows = []
    for _, row in screen_results.head(XGB_REFINE_TOP_K).iterrows():
        candidate = json.loads(row["parameters"])
        metrics = score_xgb_candidate(
            tuning_df, cv_splits, cv_years, feature_cols, candidate
        )
        refine_rows.append({
            **metrics,
            "parameters": row["parameters"],
        })
    refine_results = pd.DataFrame(refine_rows).sort_values("RMSE").reset_index(drop=True)
    best_parameters = json.loads(refine_results.iloc[0]["parameters"])
    boundary_messages = check_grid_boundaries(
        best_parameters, XGB_GRID, f"XGBoost/{block_name}"
    )
    return best_parameters, screen_results, refine_results, boundary_messages


XGB_BLOCK_PARAMS: dict[str, dict] = {}
xgb_refinement_tables = []
xgb_boundary_warnings = []

for block_name in ABLATION_SETS:
    print(f"\nTuning XGBoost for {block_name}...")
    parameters, screen_results, refine_results, boundaries = tune_xgb_block(
        block_name, FEATURE_SETS[block_name]
    )
    XGB_BLOCK_PARAMS[block_name] = parameters
    xgb_boundary_warnings += boundaries
    refine_export = refine_results.copy()
    refine_export.insert(0, "feature_set", block_name)
    xgb_refinement_tables.append(refine_export)
    screen_results.to_csv(OUTPUT_DIR / f"xgb_screen_{block_name}.csv", index=False)
    print("Selected:", parameters)
    display(refine_results.round(6))

xgb_refinement_results = pd.concat(xgb_refinement_tables, ignore_index=True)
xgb_refinement_results.to_csv(OUTPUT_DIR / "xgb_refinement_results.csv", index=False)

if xgb_boundary_warnings:
    print("\nGRID BOUNDARY WARNINGS — widen and rerun before final reporting if these remain:")
    for message in sorted(set(xgb_boundary_warnings)):
        print("  " + message)
else:
    print("\nNo selected XGBoost hyperparameter is at a grid boundary.")

`n_estimators` is treated as an ordinary hyperparameter. The validation year is used once—to score a fixed candidate—not simultaneously to choose the stopping point and estimate that candidate's error.

## 11. Development-period out-of-time comparison and specification freeze

In [ ]:
MODEL_FAMILIES = [BEST_LINEAR_FAMILY, "XGBoost"]
BLOCK_PARAMETERS = {
    block_name: {
        BEST_LINEAR_FAMILY: LINEAR_BLOCK_PARAMS[block_name],
        "XGBoost": XGB_BLOCK_PARAMS[block_name],
    }
    for block_name in ABLATION_SETS
}

# Generate a common out-of-fold prediction table for 2015-2019.
oof_blocks = []
for (train_idx, validation_idx), validation_year in zip(cv_splits, cv_years):
    fold_train = tuning_df.iloc[train_idx]
    fold_validation = tuning_df.iloc[validation_idx]
    fold_mean = float(fold_train[TARGET].mean())
    block = pd.DataFrame({
        ID_COL: fold_validation[ID_COL].to_numpy(),
        YEAR_COL: np.full(len(fold_validation), validation_year, dtype=int),
        "y": fold_validation[TARGET].to_numpy(dtype=float),
        "origin_train_mean": np.full(len(fold_validation), fold_mean, dtype=float),
    })
    for name, prediction in baseline_predictions(fold_validation, fold_mean).items():
        block[f"{name}__benchmark"] = prediction

    for block_name in ABLATION_SETS:
        feature_cols = FEATURE_SETS[block_name]
        for model_name in MODEL_FAMILIES:
            prediction, artifact = fit_predict_model(
                model_name,
                BLOCK_PARAMETERS[block_name][model_name],
                feature_cols,
                fold_train,
                fold_validation,
            )
            block[f"{model_name}__{block_name}"] = prediction
            del artifact, prediction
            gc.collect()
    oof_blocks.append(block)

development_oof = pd.concat(oof_blocks, ignore_index=True)
development_oof.to_parquet(OUTPUT_DIR / "development_oof_predictions.parquet", index=False)

PREDICTION_COLUMNS = [c for c in development_oof.columns if "__" in c]
summary_rows = []
per_year_rows = []
for column in PREDICTION_COLUMNS:
    model_name, feature_set = column.split("__", 1)
    metrics = regression_metrics(
        development_oof["y"],
        development_oof[column],
        development_oof["Mean__benchmark"],
        years=development_oof[YEAR_COL],
    )
    yearly_rmses = []
    for year, group in development_oof.groupby(YEAR_COL):
        yearly = regression_metrics(
            group["y"], group[column], group["Mean__benchmark"]
        )
        yearly_rmses.append(yearly["RMSE"])
        per_year_rows.append({
            "model": model_name,
            "feature_set": feature_set,
            "year": int(year),
            **yearly,
        })
    summary_rows.append({
        "model": model_name,
        "feature_set": feature_set,
        **metrics,
        "mean_year_RMSE": float(np.mean(yearly_rmses)),
        "worst_year_RMSE": float(np.max(yearly_rmses)),
    })

development_cv_summary = pd.DataFrame(summary_rows).sort_values("RMSE").reset_index(drop=True)
development_per_year = pd.DataFrame(per_year_rows)
development_cv_summary.to_csv(OUTPUT_DIR / "development_cv_summary.csv", index=False)
development_per_year.to_csv(OUTPUT_DIR / "development_cv_by_year.csv", index=False)

display(development_cv_summary.round(6))
print("\nRMSE by validation year:")
display(development_per_year.pivot_table(
    index="year", columns=["model", "feature_set"], values="RMSE"
).round(5))

eligible = development_cv_summary[
    development_cv_summary["feature_set"].isin(ABLATION_SETS)
].copy()
selected_row = eligible.sort_values(["RMSE", "mean_year_RMSE", "worst_year_RMSE"]).iloc[0]
FROZEN_MODEL = str(selected_row["model"])
FROZEN_FEATURE_SET = str(selected_row["feature_set"])
FROZEN_PARAMETERS = BLOCK_PARAMETERS[FROZEN_FEATURE_SET][FROZEN_MODEL]

frozen_specification = {
    "selected_on": f"expanding-window development folds {cv_years}",
    "model": FROZEN_MODEL,
    "feature_set": FROZEN_FEATURE_SET,
    "parameters": FROZEN_PARAMETERS,
    "linear_family": BEST_LINEAR_FAMILY,
    "all_block_parameters": BLOCK_PARAMETERS,
}
with open(OUTPUT_DIR / "frozen_specification.json", "w", encoding="utf-8") as file:
    json.dump(frozen_specification, file, indent=2, default=float)

print("\n" + "=" * 78)
print("SPECIFICATION FROZEN BEFORE HELD-OUT TEST")
print("=" * 78)
print(json.dumps(frozen_specification, indent=2, default=float))

The development errors are used for model selection and can therefore be mildly optimistic. The 2020–2022 block below is the independent estimate of final generalization performance. The frozen JSON file is written before any held-out prediction is produced.

### 11.1 Targeted XGBoost boundary sensitivity

The random search selected an upper-edge tree count and a low feature-subsampling rate. A boundary
flag in a random search is not proof that the optimum lies outside the grid, but the
`learning_rate × n_estimators` combination can represent an unresolved capacity limit.

The following check uses **development folds only**. It varies the two relevant dimensions around
the selected values and reports the RMSE relative to the fold-level one-standard-error band. It is
a robustness diagnostic and never replaces the frozen primary specification.


In [ ]:
xgb_boundary_sensitivity = pd.DataFrame()

if RUN_XGB_BOUNDARY_SENSITIVITY and FROZEN_MODEL == "XGBoost":
    selected_trees = int(FROZEN_PARAMETERS["n_estimators"])
    selected_colsample = float(FROZEN_PARAMETERS["colsample_bytree"])

    if selected_trees == 3_000:
        tree_values = [3_000, 4_500, 6_000]
    else:
        tree_values = sorted({
            selected_trees,
            max(selected_trees + 1, int(round(1.5 * selected_trees))),
            max(selected_trees + 2, int(round(2.0 * selected_trees))),
        })

    if np.isclose(selected_colsample, 0.30):
        colsample_values = [0.15, 0.30, 0.50]
    else:
        colsample_values = sorted({
            max(0.10, selected_colsample / 2),
            selected_colsample,
            min(1.00, selected_colsample + 0.20),
        })

    frozen_year_rmse = (
        development_per_year[
            development_per_year["model"].eq(FROZEN_MODEL)
            & development_per_year["feature_set"].eq(FROZEN_FEATURE_SET)
        ]
        .sort_values("year")["RMSE"]
        .to_numpy(dtype=float)
    )
    one_se_band = float(
        frozen_year_rmse.std(ddof=1) / np.sqrt(len(frozen_year_rmse))
    )

    sensitivity_rows = []
    for n_estimators in tree_values:
        for colsample in colsample_values:
            candidate = dict(FROZEN_PARAMETERS)
            candidate["n_estimators"] = int(n_estimators)
            candidate["colsample_bytree"] = float(colsample)
            metrics = score_xgb_candidate(
                tuning_df,
                cv_splits,
                cv_years,
                FEATURE_SETS[FROZEN_FEATURE_SET],
                candidate,
            )
            sensitivity_rows.append({
                "n_estimators": int(n_estimators),
                "colsample_bytree": float(colsample),
                "effective_capacity": float(n_estimators * candidate["learning_rate"]),
                **metrics,
                "is_original": (
                    int(n_estimators) == selected_trees
                    and np.isclose(float(colsample), selected_colsample)
                ),
            })

    xgb_boundary_sensitivity = (
        pd.DataFrame(sensitivity_rows)
        .sort_values(["RMSE", "n_estimators", "colsample_bytree"])
        .reset_index(drop=True)
    )
    original_rmse = float(
        xgb_boundary_sensitivity.loc[
            xgb_boundary_sensitivity["is_original"], "RMSE"
        ].iloc[0]
    )
    xgb_boundary_sensitivity["RMSE_change_vs_original"] = (
        xgb_boundary_sensitivity["RMSE"] - original_rmse
    )
    display(xgb_boundary_sensitivity.round(6))

    best_row = xgb_boundary_sensitivity.iloc[0]
    improvement = original_rmse - float(best_row["RMSE"])
    print(f"Fold-level 1-SE band: {one_se_band:.6f}")
    print(f"Best development-only improvement over the original setting: {improvement:.6f}")

    if improvement <= one_se_band:
        print(
            "The best extension remains inside one standard error. The boundary does not "
            "materially change the methodological conclusion; retain the pre-frozen specification."
        )
    else:
        print(
            "The extension improves development RMSE by more than one standard error. "
            "Report this as a material sensitivity, but do not use held-out outcomes to reselect the model."
        )

    xgb_boundary_sensitivity.to_csv(
        OUTPUT_DIR / "xgb_boundary_sensitivity.csv", index=False
    )
elif not RUN_XGB_BOUNDARY_SENSITIVITY:
    print(
        "RUN_XGB_BOUNDARY_SENSITIVITY=False: skipping the additional "
        "development-only XGBoost fits."
    )
else:
    print(
        "The frozen model is not XGBoost, so the frozen-parameter boundary sensitivity is not applicable."
    )


## 12. Incremental survey and text comparisons on development origins

In [ ]:
incremental_rows = []
for model_name in MODEL_FAMILIES:
    comparisons = [
        ("firm_survey", "firm", "survey given firm fundamentals"),
        ("firm_survey_text", "firm_survey", "text given firm and survey"),
    ]
    for richer, poorer, label in comparisons:
        richer_col = f"{model_name}__{richer}"
        poorer_col = f"{model_name}__{poorer}"
        mse_test = year_cluster_forecast_test(
            development_oof["y"],
            development_oof[richer_col],
            development_oof[poorer_col],
            development_oof[YEAR_COL],
            loss="squared",
        )
        mae_test = year_cluster_forecast_test(
            development_oof["y"],
            development_oof[richer_col],
            development_oof[poorer_col],
            development_oof[YEAR_COL],
            loss="absolute",
        )
        bootstrap = year_block_bootstrap_delta_rmse(
            development_oof["y"],
            development_oof[richer_col],
            development_oof[poorer_col],
            development_oof[YEAR_COL],
            n_boot=N_BOOTSTRAP,
        )
        incremental_rows.append({
            "model": model_name,
            "comparison": label,
            "richer_block": richer,
            "poorer_block": poorer,
            "mean_dMSE": mse_test["mean_loss_difference"],
            "p_squared": mse_test["p_value"],
            "dMSE_ci_low": mse_test["ci_low"],
            "dMSE_ci_high": mse_test["ci_high"],
            "mean_dMAE": mae_test["mean_loss_difference"],
            "p_absolute": mae_test["p_value"],
            "delta_RMSE": bootstrap["delta_RMSE"],
            "delta_RMSE_ci_low": bootstrap["ci_low"],
            "delta_RMSE_ci_high": bootstrap["ci_high"],
            "n_year_clusters": mse_test["n_years"],
        })

incremental_tests = pd.DataFrame(incremental_rows)
display(incremental_tests.round(7))
incremental_tests.to_csv(OUTPUT_DIR / "incremental_information_tests.csv", index=False)

print("Negative differences favour the richer information block.")
print(f"Inference is based on only {len(cv_years)} annual clusters; emphasize confidence intervals and effect size, not only p-values.")

## 13. One-time frozen held-out test

In [ ]:
final_results = pd.DataFrame()
final_predictions = pd.DataFrame()
thin_year_results = pd.DataFrame()
fitted_artifacts: dict[tuple[str, str], dict] = {}

if RUN_FINAL_TEST:
    final_train = sample_firms_to_target(train_df, FINAL_ROWS_PER_YEAR)
    final_test = test_df.copy()
    final_mean = float(final_train[TARGET].mean())

    print(f"Final training rows: {len(final_train):,} ({final_train[ID_COL].nunique():,} firms)")
    print(f"Headline test rows: {len(final_test):,}, years {HEADLINE_TEST_YEARS}")

    prediction_frame = pd.DataFrame({
        ID_COL: final_test[ID_COL].to_numpy(),
        YEAR_COL: final_test[YEAR_COL].to_numpy(),
        "y": final_test[TARGET].to_numpy(dtype=float),
    })
    for name, prediction in baseline_predictions(final_test, final_mean).items():
        prediction_frame[f"{name}__benchmark"] = prediction

    for block_name in ABLATION_SETS:
        feature_cols = FEATURE_SETS[block_name]
        for model_name in MODEL_FAMILIES:
            started = time.time()
            prediction, artifact = fit_predict_model(
                model_name,
                BLOCK_PARAMETERS[block_name][model_name],
                feature_cols,
                final_train,
                final_test,
            )
            prediction_frame[f"{model_name}__{block_name}"] = prediction
            fitted_artifacts[(model_name, block_name)] = artifact
            print(f"{model_name:12s} | {block_name:18s} | {time.time() - started:6.1f}s")
            gc.collect()

    final_predictions = prediction_frame
    final_predictions.to_parquet(OUTPUT_DIR / "held_out_test_predictions.parquet", index=False)

    result_rows = []
    final_prediction_cols = [c for c in final_predictions.columns if "__" in c]
    for column in final_prediction_cols:
        model_name, feature_set = column.split("__", 1)
        pooled = regression_metrics(
            final_predictions["y"],
            final_predictions[column],
            final_predictions["Mean__benchmark"],
            years=final_predictions[YEAR_COL],
        )
        result_rows.append({
            "scope": "pooled",
            "year": f"{min(HEADLINE_TEST_YEARS)}-{max(HEADLINE_TEST_YEARS)}",
            "model": model_name,
            "feature_set": feature_set,
            **pooled,
        })
        for year, group in final_predictions.groupby(YEAR_COL):
            yearly = regression_metrics(
                group["y"], group[column], group["Mean__benchmark"]
            )
            result_rows.append({
                "scope": "year",
                "year": int(year),
                "model": model_name,
                "feature_set": feature_set,
                **yearly,
            })

    final_results = pd.DataFrame(result_rows)
    final_results.to_csv(OUTPUT_DIR / "held_out_test_results.csv", index=False)

    pooled = final_results[final_results["scope"].eq("pooled")].set_index(["model", "feature_set"])
    frozen_row = pooled.loc[(FROZEN_MODEL, FROZEN_FEATURE_SET)]
    print("\nPRIMARY RESULT — specification selected before seeing the test:")
    display(pd.DataFrame([{
        "model": FROZEN_MODEL,
        "feature_set": FROZEN_FEATURE_SET,
        "RMSE": frozen_row["RMSE"],
        "MAE": frozen_row["MAE"],
        "R2_oos": frozen_row["R2_oos"],
        "R2_oos_within_year": frozen_row["R2_oos_within_year"],
        "n": frozen_row["n"],
    }]).round(6))

    print("All pooled held-out rows (descriptive; do not reselect the model from this table):")
    display(final_results[final_results["scope"].eq("pooled")].sort_values("RMSE").round(6))
    print("Year-specific RMSE:")
    display(final_results[final_results["scope"].eq("year")].pivot_table(
        index="year", columns=["model", "feature_set"], values="RMSE"
    ).round(5))

    test_best = final_results[
        final_results["scope"].eq("pooled")
        & final_results["feature_set"].isin(ABLATION_SETS)
    ].sort_values("RMSE").iloc[0]
    if (test_best["model"], test_best["feature_set"]) != (FROZEN_MODEL, FROZEN_FEATURE_SET):
        print(
            "The numerically best test row differs from the frozen specification. "
            "It must remain a descriptive observation, not replace the primary model."
        )

    # Thin/incomplete years are scored only as a sensitivity check.
    if not thin_test_df.empty:
        thin_predictions = pd.DataFrame({
            ID_COL: thin_test_df[ID_COL].to_numpy(),
            YEAR_COL: thin_test_df[YEAR_COL].to_numpy(),
            "y": thin_test_df[TARGET].to_numpy(dtype=float),
        })
        for name, prediction in baseline_predictions(thin_test_df, final_mean).items():
            thin_predictions[f"{name}__benchmark"] = prediction
        for block_name in ABLATION_SETS:
            feature_cols = FEATURE_SETS[block_name]
            for model_name in MODEL_FAMILIES:
                artifact = fitted_artifacts[(model_name, block_name)]
                thin_predictions[f"{model_name}__{block_name}"] = predict_fitted_artifact(
                    artifact, model_name, feature_cols, thin_test_df
                )

        thin_rows = []
        for column in [c for c in thin_predictions.columns if "__" in c]:
            model_name, feature_set = column.split("__", 1)
            for year, group in thin_predictions.groupby(YEAR_COL):
                metrics = regression_metrics(
                    group["y"], group[column], group["Mean__benchmark"]
                )
                thin_rows.append({
                    "year": int(year),
                    "model": model_name,
                    "feature_set": feature_set,
                    **metrics,
                })
        thin_year_results = pd.DataFrame(thin_rows)
        thin_year_results.to_csv(OUTPUT_DIR / "thin_year_sensitivity.csv", index=False)
        print("\nThin/incomplete-year sensitivity — excluded from headline pooling:")
        display(thin_year_results.pivot_table(
            index="year", columns=["model", "feature_set"], values="RMSE"
        ).round(5))
else:
    print("RUN_FINAL_TEST=False: the held-out period was not evaluated.")

### 13.1 Incremental survey and text value on the untouched held-out block

Section 12 compares nested information blocks on development folds. The table below repeats the
same contrasts on 2020–2022, which was not used for tuning. With only three annual clusters, the
confidence intervals are necessarily imprecise. The purpose is to compare signs and magnitudes
across regimes, not to reselect the model.


In [ ]:
incremental_comparison = pd.DataFrame()

if RUN_FINAL_TEST and RUN_HELDOUT_INCREMENT_TESTS:
    def incremental_information_table(
        frame: pd.DataFrame,
        sample_label: str,
    ) -> pd.DataFrame:
        comparisons = [
            ("firm_survey", "firm", "survey given firm fundamentals"),
            ("firm_survey_text", "firm_survey", "text given firm and survey"),
            ("firm_survey_text", "firm", "survey plus text given firm fundamentals"),
        ]
        rows = []
        for model_name in MODEL_FAMILIES:
            for richer, poorer, label in comparisons:
                richer_col = f"{model_name}__{richer}"
                poorer_col = f"{model_name}__{poorer}"
                if richer_col not in frame.columns or poorer_col not in frame.columns:
                    continue
                squared = year_cluster_forecast_test(
                    frame["y"], frame[richer_col], frame[poorer_col],
                    frame[YEAR_COL], loss="squared"
                )
                absolute = year_cluster_forecast_test(
                    frame["y"], frame[richer_col], frame[poorer_col],
                    frame[YEAR_COL], loss="absolute"
                )
                bootstrap = year_block_bootstrap_delta_rmse(
                    frame["y"], frame[richer_col], frame[poorer_col],
                    frame[YEAR_COL], n_boot=N_BOOTSTRAP
                )
                yearly_differential = (
                    pd.DataFrame({
                        YEAR_COL: frame[YEAR_COL].to_numpy(),
                        "d": (
                            (frame["y"] - frame[richer_col]) ** 2
                            - (frame["y"] - frame[poorer_col]) ** 2
                        ).to_numpy(),
                    })
                    .groupby(YEAR_COL)["d"]
                    .mean()
                )
                rows.append({
                    "sample": sample_label,
                    "model": model_name,
                    "comparison": label,
                    "delta_RMSE": bootstrap["delta_RMSE"],
                    "delta_RMSE_ci_low": bootstrap["ci_low"],
                    "delta_RMSE_ci_high": bootstrap["ci_high"],
                    "mean_dMSE": squared["mean_loss_difference"],
                    "p_squared": squared["p_value"],
                    "mean_dMAE": absolute["mean_loss_difference"],
                    "p_absolute": absolute["p_value"],
                    "years_favouring_richer": (
                        f"{int((yearly_differential < 0).sum())}/{len(yearly_differential)}"
                    ),
                    "n_year_clusters": squared["n_years"],
                })
        return pd.DataFrame(rows)

    development_increments = incremental_information_table(
        development_oof, "development 2015-2019"
    )
    held_out_increments = incremental_information_table(
        final_predictions, "held-out 2020-2022"
    )
    incremental_comparison = pd.concat(
        [development_increments, held_out_increments], ignore_index=True
    )
    display(incremental_comparison.round(6))

    sign_check = (
        incremental_comparison
        .pivot_table(
            index=["model", "comparison"],
            columns="sample",
            values="delta_RMSE",
        )
        .reset_index()
    )
    development_label = "development 2015-2019"
    held_out_label = "held-out 2020-2022"
    if {development_label, held_out_label}.issubset(sign_check.columns):
        sign_check["sign_agrees"] = (
            np.sign(sign_check[development_label])
            == np.sign(sign_check[held_out_label])
        )
        print("Sign stability across evaluation regimes:")
        display(sign_check.round(6))

    incremental_comparison.to_csv(
        OUTPUT_DIR / "incremental_information_both_samples.csv", index=False
    )
else:
    print("Held-out incremental tests skipped.")


### 13.2 Research question 2: heterogeneity across firm characteristics

The predictive value of survey and text information is evaluated separately by firm size,
leverage, age, and—where sample size permits—two-digit industry. Tercile cut points are estimated
from the development period only.

Negative `delta_RMSE` means that the richer information block performs better. The pairwise
contrast table asks whether the richer block helps the high group more than the low group. With
five development years and three held-out years, all inference should be described as exploratory.


In [ ]:
rq2_results = pd.DataFrame()
rq2_group_contrasts = pd.DataFrame()

if RUN_FINAL_TEST and RUN_HETEROGENEITY_ANALYSIS:
    grouping_specifications = [
        ("size", "log_toas", ["small", "medium", "large"], "large", "small"),
        ("leverage", "liabilities_to_assets", ["low", "mid", "high"], "high", "low"),
        ("age", "firm_age", ["young", "middle", "mature"], "mature", "young"),
    ]
    grouping_specifications = [
        spec for spec in grouping_specifications if spec[1] in df.columns
    ]

    characteristic_columns = list(dict.fromkeys(
        [column for _, column, _, _, _ in grouping_specifications]
        + (["naics_2digit"] if "naics_2digit" in df.columns else [])
    ))
    characteristics = (
        df[[ID_COL, YEAR_COL] + characteristic_columns]
        .drop_duplicates([ID_COL, YEAR_COL])
    )

    development_cut_points = {}
    for group_name, column, labels, high_label, low_label in grouping_specifications:
        values = pd.to_numeric(train_df[column], errors="coerce")
        cut_points = np.unique(values.quantile([1 / 3, 2 / 3]).dropna().to_numpy())
        if len(cut_points) == 2:
            development_cut_points[group_name] = cut_points

    def attach_groups(frame: pd.DataFrame) -> pd.DataFrame:
        merged = frame.merge(
            characteristics, on=[ID_COL, YEAR_COL], how="left", validate="one_to_one"
        )
        for group_name, column, labels, _, _ in grouping_specifications:
            cut_points = development_cut_points.get(group_name)
            if cut_points is None:
                continue
            merged[f"{group_name}_group"] = pd.cut(
                pd.to_numeric(merged[column], errors="coerce"),
                bins=[-np.inf, *cut_points, np.inf],
                labels=labels,
                include_lowest=True,
            )
        return merged

    comparisons = [
        ("firm_survey", "firm", "survey given firm"),
        ("firm_survey_text", "firm_survey", "text given firm and survey"),
        ("firm_survey_text", "firm", "survey plus text given firm"),
    ]

    def subgroup_incremental_value(
        frame: pd.DataFrame,
        group_column: str,
        model_name: str,
        sample_label: str,
        minimum_rows: int,
    ) -> pd.DataFrame:
        rows = []
        for richer, poorer, comparison_label in comparisons:
            richer_col = f"{model_name}__{richer}"
            poorer_col = f"{model_name}__{poorer}"
            if richer_col not in frame.columns or poorer_col not in frame.columns:
                continue
            for group, part in frame.groupby(group_column, observed=True):
                if len(part) < minimum_rows:
                    continue
                bootstrap = year_block_bootstrap_delta_rmse(
                    part["y"], part[richer_col], part[poorer_col],
                    part[YEAR_COL], n_boot=N_BOOTSTRAP
                )
                squared = year_cluster_forecast_test(
                    part["y"], part[richer_col], part[poorer_col],
                    part[YEAR_COL], loss="squared"
                )
                baseline = part["Mean__benchmark"]
                rows.append({
                    "sample": sample_label,
                    "model": model_name,
                    "comparison": comparison_label,
                    "grouping": group_column.replace("_group", ""),
                    "group": str(group),
                    "n": len(part),
                    "RMSE_poorer": regression_metrics(
                        part["y"], part[poorer_col], baseline
                    )["RMSE"],
                    "RMSE_richer": regression_metrics(
                        part["y"], part[richer_col], baseline
                    )["RMSE"],
                    "delta_RMSE": bootstrap["delta_RMSE"],
                    "ci_low": bootstrap["ci_low"],
                    "ci_high": bootstrap["ci_high"],
                    "mean_dMSE": squared["mean_loss_difference"],
                    "p_value": squared["p_value"],
                    "n_year_clusters": squared["n_years"],
                })
        return pd.DataFrame(rows)

    def endpoint_group_contrast(
        frame: pd.DataFrame,
        group_column: str,
        high_group: str,
        low_group: str,
        model_name: str,
        sample_label: str,
    ) -> pd.DataFrame:
        rows = []
        for richer, poorer, comparison_label in comparisons:
            richer_col = f"{model_name}__{richer}"
            poorer_col = f"{model_name}__{poorer}"
            if richer_col not in frame.columns or poorer_col not in frame.columns:
                continue
            work = frame[[YEAR_COL, group_column, "y", richer_col, poorer_col]].dropna()
            work["loss_difference"] = (
                (work["y"] - work[richer_col]) ** 2
                - (work["y"] - work[poorer_col]) ** 2
            )
            annual = (
                work.groupby([YEAR_COL, group_column], observed=True)["loss_difference"]
                .mean()
                .unstack(group_column)
            )
            if high_group not in annual.columns or low_group not in annual.columns:
                continue
            contrast = (annual[high_group] - annual[low_group]).dropna()
            g = len(contrast)
            point = float(contrast.mean()) if g else np.nan
            if g >= 2 and float(contrast.std(ddof=1)) > 0:
                se = float(contrast.std(ddof=1) / np.sqrt(g))
                critical = float(stats.t.ppf(0.975, df=g - 1))
                p_value = float(2 * stats.t.sf(abs(point / se), df=g - 1))
                ci_low, ci_high = point - critical * se, point + critical * se
            else:
                p_value = ci_low = ci_high = np.nan
            rows.append({
                "sample": sample_label,
                "model": model_name,
                "comparison": comparison_label,
                "grouping": group_column.replace("_group", ""),
                "contrast": f"{high_group} minus {low_group}",
                "mean_difference_in_dMSE": point,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "p_value": p_value,
                "n_year_clusters": g,
            })
        return pd.DataFrame(rows)

    grouped_samples = [
        (attach_groups(development_oof), "development 2015-2019", 1_000),
        (attach_groups(final_predictions), "held-out 2020-2022", 1_000),
    ]

    subgroup_tables = []
    contrast_tables = []
    for grouped_frame, sample_label, minimum_rows in grouped_samples:
        for model_name in MODEL_FAMILIES:
            for group_name, _, labels, high_label, low_label in grouping_specifications:
                group_column = f"{group_name}_group"
                if group_column not in grouped_frame.columns:
                    continue
                table = subgroup_incremental_value(
                    grouped_frame, group_column, model_name,
                    sample_label, minimum_rows
                )
                if not table.empty:
                    subgroup_tables.append(table)
                contrast = endpoint_group_contrast(
                    grouped_frame, group_column, high_label, low_label,
                    model_name, sample_label
                )
                if not contrast.empty:
                    contrast_tables.append(contrast)

    if subgroup_tables:
        rq2_results = pd.concat(subgroup_tables, ignore_index=True)
        print("Group-specific incremental value (negative delta_RMSE favours richer information):")
        display(rq2_results.round(6))
        rq2_results.to_csv(
            OUTPUT_DIR / "rq2_subgroup_incremental_value.csv", index=False
        )

    if contrast_tables:
        rq2_group_contrasts = pd.concat(contrast_tables, ignore_index=True)
        print("Between-group contrasts in squared-loss improvement:")
        display(rq2_group_contrasts.round(6))
        rq2_group_contrasts.to_csv(
            OUTPUT_DIR / "rq2_between_group_contrasts.csv", index=False
        )

    if "naics_2digit" in characteristic_columns:
        sector_frame = attach_groups(final_predictions)
        sector_tables = []
        for model_name in ["XGBoost"]:
            sector = subgroup_incremental_value(
                sector_frame, "naics_2digit", model_name,
                "held-out 2020-2022", minimum_rows=5_000
            )
            if not sector.empty:
                sector_tables.append(sector)
        if sector_tables:
            rq2_sector_results = pd.concat(sector_tables, ignore_index=True)
            print("Two-digit sectors with at least 5,000 held-out observations:")
            display(rq2_sector_results.sort_values(
                ["comparison", "delta_RMSE"]
            ).round(6))
            rq2_sector_results.to_csv(
                OUTPUT_DIR / "rq2_sector_incremental_value.csv", index=False
            )

    # One compact plot: XGBoost, full macro block versus firm-only, held-out.
    if not rq2_results.empty:
        plot_data = rq2_results[
            rq2_results["sample"].eq("held-out 2020-2022")
            & rq2_results["model"].eq("XGBoost")
            & rq2_results["comparison"].eq("survey plus text given firm")
        ].copy()
        if not plot_data.empty:
            plot_data["label"] = (
                plot_data["grouping"] + " | " + plot_data["group"]
            )
            positions = np.arange(len(plot_data))
            fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(plot_data))))
            ax.errorbar(
                plot_data["delta_RMSE"],
                positions,
                xerr=[
                    plot_data["delta_RMSE"] - plot_data["ci_low"],
                    plot_data["ci_high"] - plot_data["delta_RMSE"],
                ],
                fmt="o",
                capsize=3,
            )
            ax.axvline(0, linewidth=1)
            ax.set_yticks(positions)
            ax.set_yticklabels(plot_data["label"], fontsize=8)
            ax.set_xlabel(
                "Change in RMSE from survey + text (negative favours richer block)"
            )
            ax.set_title("Held-out heterogeneity in macro-information value")
            ax.invert_yaxis()
            plt.tight_layout()
            plt.savefig(
                OUTPUT_DIR / "rq2_subgroup_incremental_value.png", dpi=150
            )
            plt.show()
else:
    print("Heterogeneity analysis skipped.")


### 13.3 Robustness: previously observed versus new firms

Most held-out firms also occur in the development period. The table below separates firms with at
least one pre-2020 observation from firms first appearing in the held-out block. This distinguishes
future prediction for known firms from generalization to entirely new firms.


In [ ]:
seen_new_firm_results = pd.DataFrame()

if RUN_FINAL_TEST and RUN_SEEN_NEW_FIRM_ROBUSTNESS:
    seen_frame = final_predictions.copy()
    seen_frame["firm_status"] = np.where(
        seen_frame[ID_COL].isin(train_firms),
        "seen in development",
        "new in held-out",
    )

    rows = []
    prediction_columns = [
        c for c in seen_frame.columns
        if "__" in c and c not in {ID_COL, YEAR_COL}
    ]
    for firm_status, part in seen_frame.groupby("firm_status"):
        benchmark_rmse = regression_metrics(
            part["y"], part["Mean__benchmark"], part["Mean__benchmark"]
        )["RMSE"]
        for column in prediction_columns:
            model_name, feature_set = column.split("__", 1)
            metrics = regression_metrics(
                part["y"],
                part[column],
                part["Mean__benchmark"],
                years=part[YEAR_COL],
            )
            rows.append({
                "firm_status": firm_status,
                "model": model_name,
                "feature_set": feature_set,
                "n": len(part),
                "benchmark_RMSE": benchmark_rmse,
                "RMSE_improvement_vs_mean": benchmark_rmse - metrics["RMSE"],
                **metrics,
            })

    seen_new_firm_results = (
        pd.DataFrame(rows)
        .sort_values(["firm_status", "RMSE"])
        .reset_index(drop=True)
    )
    display(seen_new_firm_results.round(6))

    frozen_seen_new = seen_new_firm_results[
        seen_new_firm_results["model"].eq(FROZEN_MODEL)
        & seen_new_firm_results["feature_set"].eq(FROZEN_FEATURE_SET)
    ]
    print("Frozen specification by firm status:")
    display(frozen_seen_new.round(6))

    seen_new_firm_results.to_csv(
        OUTPUT_DIR / "seen_vs_new_firm_robustness.csv", index=False
    )
else:
    print("Seen-versus-new-firm robustness skipped.")


### 13.4 Robustness: fold-fitted clipping for XGBoost

The primary XGBoost specification uses raw engineered ratios. This check applies the same 1st/99th
percentile clipping rule used for linear models, but estimates the clipping bounds separately
inside each development fold and once on the final development sample. Hyperparameters are held
fixed. The result is a robustness comparison, not a second model-selection exercise.


In [ ]:
clipped_xgb_robustness = pd.DataFrame()

if RUN_FINAL_TEST and RUN_CLIPPED_XGB_ROBUSTNESS:
    best_xgb_row = (
        development_cv_summary[
            development_cv_summary["model"].eq("XGBoost")
            & development_cv_summary["feature_set"].isin(ABLATION_SETS)
        ]
        .sort_values("RMSE")
        .iloc[0]
    )
    clipped_block = str(best_xgb_row["feature_set"])
    clipped_parameters = XGB_BLOCK_PARAMS[clipped_block]
    clipped_features = FEATURE_SETS[clipped_block]

    def fit_xgb_clip_bounds(
        train_frame: pd.DataFrame,
        feature_cols: list[str],
    ) -> dict[str, tuple[float, float]]:
        numeric = [c for c in feature_cols if c not in CATEGORY_COLS]
        bounds = {}
        for column in numeric:
            values = pd.to_numeric(train_frame[column], errors="coerce")
            lower = float(values.quantile(WINSOR_LOWER))
            upper = float(values.quantile(WINSOR_UPPER))
            if np.isfinite(lower) and np.isfinite(upper):
                bounds[column] = (lower, upper)
        return bounds

    def prepare_xgb_frame_clipped(
        frame: pd.DataFrame,
        feature_cols: list[str],
        category_levels: dict[str, pd.Index],
        bounds: dict[str, tuple[float, float]],
    ) -> pd.DataFrame:
        work = frame[feature_cols].copy()
        for column, (lower, upper) in bounds.items():
            work[column] = pd.to_numeric(
                work[column], errors="coerce"
            ).clip(lower, upper)
        return prepare_xgb_frame(work, feature_cols, category_levels)

    def fit_predict_xgb_clipped(
        train_frame: pd.DataFrame,
        score_frame: pd.DataFrame,
    ):
        levels = tree_category_levels(train_frame, clipped_features)
        bounds = fit_xgb_clip_bounds(train_frame, clipped_features)
        X_train = prepare_xgb_frame_clipped(
            train_frame, clipped_features, levels, bounds
        )
        X_score = prepare_xgb_frame_clipped(
            score_frame, clipped_features, levels, bounds
        )
        model = make_xgb_estimator(clipped_parameters)
        model.fit(X_train, train_frame[TARGET], verbose=False)
        prediction = np.asarray(model.predict(X_score), dtype=float)
        del X_train, X_score
        return prediction, {"model": model, "category_levels": levels, "bounds": bounds}

    # Development out-of-fold predictions with fold-specific clipping.
    clipped_oof_parts = []
    for (train_idx, validation_idx), validation_year in zip(cv_splits, cv_years):
        fold_train = tuning_df.iloc[train_idx]
        fold_validation = tuning_df.iloc[validation_idx]
        prediction, artifact = fit_predict_xgb_clipped(
            fold_train, fold_validation
        )
        clipped_oof_parts.append(pd.DataFrame({
            ID_COL: fold_validation[ID_COL].to_numpy(),
            YEAR_COL: fold_validation[YEAR_COL].to_numpy(),
            "y": fold_validation[TARGET].to_numpy(dtype=float),
            "Mean__benchmark": np.full(
                len(fold_validation), float(fold_train[TARGET].mean())
            ),
            "XGBoost_clipped": prediction,
        }))
        del artifact
        gc.collect()

    clipped_oof = pd.concat(clipped_oof_parts, ignore_index=True)
    original_oof = development_oof[
        [ID_COL, YEAR_COL, f"XGBoost__{clipped_block}"]
    ]
    clipped_oof = clipped_oof.merge(
        original_oof, on=[ID_COL, YEAR_COL], how="left", validate="one_to_one"
    )

    # Held-out predictions with bounds learned on the full development sample.
    clipped_test_prediction, clipped_final_artifact = fit_predict_xgb_clipped(
        final_train, final_test
    )
    clipped_test = final_predictions[
        [ID_COL, YEAR_COL, "y", "Mean__benchmark", f"XGBoost__{clipped_block}"]
    ].copy()
    clipped_test["XGBoost_clipped"] = clipped_test_prediction

    robustness_rows = []
    for sample_label, frame in [
        ("development 2015-2019", clipped_oof),
        ("held-out 2020-2022", clipped_test),
    ]:
        original_col = f"XGBoost__{clipped_block}"
        for specification, prediction_col in [
            ("primary raw XGBoost", original_col),
            ("fold-fitted clipped XGBoost", "XGBoost_clipped"),
        ]:
            metrics = regression_metrics(
                frame["y"], frame[prediction_col],
                frame["Mean__benchmark"], years=frame[YEAR_COL]
            )
            robustness_rows.append({
                "sample": sample_label,
                "feature_set": clipped_block,
                "specification": specification,
                **metrics,
            })
        comparison = year_block_bootstrap_delta_rmse(
            frame["y"],
            frame["XGBoost_clipped"],
            frame[original_col],
            frame[YEAR_COL],
            n_boot=N_BOOTSTRAP,
        )
        robustness_rows.append({
            "sample": sample_label,
            "feature_set": clipped_block,
            "specification": "clipped minus primary delta",
            "RMSE": comparison["delta_RMSE"],
            "MAE": np.nan,
            "R2_oos": np.nan,
            "R2_oos_within_year": np.nan,
            "n": len(frame),
            "ci_low": comparison["ci_low"],
            "ci_high": comparison["ci_high"],
        })

    clipped_xgb_robustness = pd.DataFrame(robustness_rows)
    display(clipped_xgb_robustness.round(6))
    clipped_xgb_robustness.to_csv(
        OUTPUT_DIR / "clipped_xgb_robustness.csv", index=False
    )
else:
    print("Fold-fitted XGBoost clipping robustness skipped.")


### 13.5 Reporting diagnostics

These summaries make the loss-function trade-off, development-to-test shrinkage, benchmark
behaviour, and thin-year sensitivity explicit. They do not alter model selection.


In [ ]:
reporting_diagnostics = {}

if RUN_FINAL_TEST and not final_results.empty:
    pooled_held_out = (
        final_results[final_results["scope"].eq("pooled")]
        .set_index(["model", "feature_set"])
    )
    pooled_development = development_cv_summary.set_index(
        ["model", "feature_set"]
    )
    frozen_key = (FROZEN_MODEL, FROZEN_FEATURE_SET)

    shrinkage = pd.DataFrame([
        {
            "metric": metric,
            "development": float(pooled_development.loc[frozen_key, metric]),
            "held_out": float(pooled_held_out.loc[frozen_key, metric]),
        }
        for metric in ("R2_oos", "R2_oos_within_year", "RMSE", "MAE")
    ])
    shrinkage["held_out_minus_development"] = (
        shrinkage["held_out"] - shrinkage["development"]
    )
    print("Development-to-held-out comparison:")
    display(shrinkage.round(6))

    model_rows = pooled_held_out[
        pooled_held_out.index.get_level_values("feature_set").isin(ABLATION_SETS)
    ]
    mean_mae = float(pooled_held_out.loc[("Mean", "benchmark"), "MAE"])
    loss_comparison = (
        model_rows[["RMSE", "MAE", "R2_oos"]]
        .assign(beats_mean_on_MAE=lambda x: x["MAE"] < mean_mae)
        .sort_values("RMSE")
    )
    print("Squared versus absolute loss:")
    display(loss_comparison.round(6))

    benchmark_view = pooled_held_out[
        pooled_held_out.index.get_level_values("feature_set").eq("benchmark")
    ][["RMSE", "MAE", "R2_oos"]]
    print("Naive benchmarks:")
    display(benchmark_view.round(6))

    if not thin_year_results.empty:
        print("Thin-year sensitivity:")
        display(thin_year_results.round(6))

    shrinkage.to_csv(
        OUTPUT_DIR / "development_to_heldout_shrinkage.csv", index=False
    )
    loss_comparison.to_csv(
        OUTPUT_DIR / "squared_versus_absolute_loss.csv"
    )
    reporting_diagnostics = {
        "shrinkage": shrinkage,
        "loss_comparison": loss_comparison,
        "benchmarks": benchmark_view,
    }
else:
    print("Reporting diagnostics skipped.")


## 14. Interpretation of the frozen models

In [ ]:
linear_coefficient_table = pd.DataFrame()
xgb_gain_importance = pd.DataFrame()
xgb_permutation_importance = pd.DataFrame()

if RUN_FINAL_TEST and fitted_artifacts:
    # Interpret the best development-selected linear block, regardless of the overall winner.
    best_linear_row = development_cv_summary[
        (development_cv_summary["model"] == BEST_LINEAR_FAMILY)
        & development_cv_summary["feature_set"].isin(ABLATION_SETS)
    ].sort_values("RMSE").iloc[0]
    linear_block = str(best_linear_row["feature_set"])
    linear_artifact = fitted_artifacts[(BEST_LINEAR_FAMILY, linear_block)]
    linear_model = linear_artifact["model"]
    estimator = linear_model.named_steps["model"]
    if BEST_LINEAR_FAMILY == "Ridge":
        converged = True
    else:
        n_iter = int(np.max(np.atleast_1d(estimator.n_iter_)))
        converged = n_iter < estimator.max_iter

    if converged:
        names = linear_model.named_steps["preprocess"].get_feature_names_out()
        linear_coefficient_table = (
            pd.DataFrame({"feature": names, "coefficient": estimator.coef_})
            .assign(abs_coefficient=lambda x: x["coefficient"].abs())
            .sort_values("abs_coefficient", ascending=False)
            .reset_index(drop=True)
        )
        print(f"Linear coefficients: {BEST_LINEAR_FAMILY} | {linear_block}")
        display(linear_coefficient_table.head(25).round(6))
        linear_coefficient_table.to_csv(OUTPUT_DIR / "linear_coefficients.csv", index=False)
    else:
        print("The final linear model did not converge; coefficients are not reported.")

    # Interpret XGBoost only for the feature block selected for XGBoost on development CV.
    best_xgb_row = development_cv_summary[
        (development_cv_summary["model"] == "XGBoost")
        & development_cv_summary["feature_set"].isin(ABLATION_SETS)
    ].sort_values("RMSE").iloc[0]
    xgb_block = str(best_xgb_row["feature_set"])
    xgb_artifact = fitted_artifacts[("XGBoost", xgb_block)]
    xgb_model = xgb_artifact["model"]
    gain = xgb_model.get_booster().get_score(importance_type="gain")
    xgb_gain_importance = (
        pd.DataFrame({"feature": list(gain.keys()), "gain": list(gain.values())})
        .sort_values("gain", ascending=False)
        .reset_index(drop=True)
    )
    print(f"XGBoost gain importance: {xgb_block}")
    display(xgb_gain_importance.head(25).round(6))
    xgb_gain_importance.to_csv(OUTPUT_DIR / "xgb_gain_importance.csv", index=False)

    if RUN_PERMUTATION_IMPORTANCE:
        sample_n = min(PERMUTATION_SAMPLE_SIZE, len(test_df))
        importance_sample = test_df.sample(sample_n, random_state=RANDOM_STATE)
        X_importance = prepare_xgb_frame(
            importance_sample,
            FEATURE_SETS[xgb_block],
            xgb_artifact["category_levels"],
        )
        permutation = permutation_importance(
            xgb_model,
            X_importance,
            importance_sample[TARGET],
            scoring="neg_root_mean_squared_error",
            n_repeats=PERMUTATION_REPEATS,
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
        xgb_permutation_importance = (
            pd.DataFrame({
                "feature": FEATURE_SETS[xgb_block],
                "importance_mean": permutation.importances_mean,
                "importance_std": permutation.importances_std,
            })
            .sort_values("importance_mean", ascending=False)
            .reset_index(drop=True)
        )
        print("Held-out permutation importance (post-evaluation descriptive analysis):")
        display(xgb_permutation_importance.head(25).round(6))
        xgb_permutation_importance.to_csv(
            OUTPUT_DIR / "xgb_permutation_importance.csv", index=False
        )
        del X_importance, importance_sample
        gc.collect()

Coefficient and importance tables describe predictive associations, not causal effects. Standardizing the one-hot indicators makes penalized-regression coefficient magnitudes more comparable than in the earlier notebook, although correlated predictors can still share or redistribute importance.

### 14.1 Model-implied macro sensitivity

The selected primary block may contain no macro variables, so feature importance alone cannot
answer how predictions move with expectations. The first diagnostic below replaces the complete
survey-and-text vector with historical annual vectors. This preserves the observed covariance
among macro indicators and is preferable to varying one highly correlated series in isolation.

A second, one-variable perturbation is reported only for a small set of indicators. It is a
synthetic predictive sensitivity exercise—not a causal effect, structural elasticity, or
necessarily feasible macroeconomic scenario.


In [ ]:
macro_sensitivity_results = pd.DataFrame()
macro_one_variable_sensitivity = pd.DataFrame()

if RUN_FINAL_TEST and RUN_MACRO_SENSITIVITY:
    macro_block = "firm_survey_text"
    macro_artifact = fitted_artifacts.get(("XGBoost", macro_block))
    if macro_artifact is None:
        print("The fitted XGBoost full-information artifact is unavailable.")
    else:
        macro_features = FEATURE_SETS[macro_block]
        available_macro_features = [
            c for c in list(dict.fromkeys(SURVEY_FEATURES + TEXT_FEATURES))
            if c in macro_features
        ]
        sensitivity_sample = final_test.sample(
            min(20_000, len(final_test)),
            random_state=RANDOM_STATE,
        )
        target_sd = float(final_test[TARGET].std())

        # Coherent historical scenarios: replace the full macro vector together.
        annual_macro = (
            final_train.groupby(YEAR_COL)[available_macro_features]
            .median(numeric_only=True)
            .sort_index()
        )
        scenario_rows = []
        for scenario_year, scenario in annual_macro.iterrows():
            work = sensitivity_sample.copy()
            for feature in available_macro_features:
                value = scenario.get(feature, np.nan)
                work[feature] = value
            prediction = predict_fitted_artifact(
                macro_artifact, "XGBoost", macro_features, work
            )
            scenario_rows.append({
                "historical_macro_scenario_year": int(scenario_year),
                "mean_predicted_growth": float(np.mean(prediction)),
                "median_predicted_growth": float(np.median(prediction)),
            })

        macro_sensitivity_results = pd.DataFrame(scenario_rows)
        scenario_range = (
            macro_sensitivity_results["mean_predicted_growth"].max()
            - macro_sensitivity_results["mean_predicted_growth"].min()
        )
        print("Coherent historical macro scenarios:")
        display(macro_sensitivity_results.round(6))
        print(
            f"Range in mean prediction across historical macro vectors: "
            f"{scenario_range:.6f} ({scenario_range / target_sd:.3%} of held-out target SD)."
        )

        fig, ax = plt.subplots(figsize=(9, 4.5))
        ax.plot(
            macro_sensitivity_results["historical_macro_scenario_year"],
            macro_sensitivity_results["mean_predicted_growth"],
            marker="o",
        )
        ax.set_xlabel("Historical macro-vector year imposed on held-out firms")
        ax.set_ylabel("Mean predicted next-year NCLI growth")
        ax.set_title("Model-implied sensitivity to coherent historical macro states")
        plt.tight_layout()
        plt.savefig(
            OUTPUT_DIR / "macro_historical_scenario_sensitivity.png", dpi=150
        )
        plt.show()

        # One-variable perturbations: explicitly synthetic and non-causal.
        candidates = [
            c for c in [
                "ifo_business_expectations",
                "de_economic_sentiment_index",
                "lm_polarity",
                "uncertainty_ratio",
            ]
            if c in available_macro_features
        ]
        one_variable_rows = []
        for feature in candidates:
            observed = pd.to_numeric(final_train[feature], errors="coerce").dropna()
            if observed.empty:
                continue
            unique_values = np.sort(observed.unique())
            grid_values = (
                unique_values
                if len(unique_values) <= 15
                else np.unique(np.quantile(observed, np.linspace(0.05, 0.95, 11)))
            )
            feature_predictions = []
            for value in grid_values:
                work = sensitivity_sample.copy()
                work[feature] = value
                prediction = predict_fitted_artifact(
                    macro_artifact, "XGBoost", macro_features, work
                )
                feature_predictions.append(float(np.mean(prediction)))
                one_variable_rows.append({
                    "feature": feature,
                    "value": float(value),
                    "mean_prediction": float(np.mean(prediction)),
                })
            prediction_range = max(feature_predictions) - min(feature_predictions)
            print(
                f"{feature}: synthetic prediction range {prediction_range:.6f} "
                f"({prediction_range / target_sd:.3%} of held-out target SD)"
            )

        macro_one_variable_sensitivity = pd.DataFrame(one_variable_rows)
        if not macro_one_variable_sensitivity.empty:
            display(macro_one_variable_sensitivity.round(6))

        macro_sensitivity_results.to_csv(
            OUTPUT_DIR / "macro_historical_scenario_sensitivity.csv", index=False
        )
        macro_one_variable_sensitivity.to_csv(
            OUTPUT_DIR / "macro_one_variable_sensitivity.csv", index=False
        )
else:
    print("Macro-sensitivity analysis skipped.")


## 15. Plots

In [ ]:
plot_dev = development_cv_summary[
    development_cv_summary["feature_set"].isin(ABLATION_SETS)
].copy()
plot_dev["label"] = plot_dev["model"] + " | " + plot_dev["feature_set"]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_dev["label"], plot_dev["RMSE"])
ax.set_xlabel("Development expanding-window RMSE")
ax.set_title("Model and information-set comparison before the frozen test")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

if RUN_FINAL_TEST and not final_results.empty:
    pooled = final_results[
        final_results["scope"].eq("pooled")
        & final_results["feature_set"].isin(ABLATION_SETS)
    ].copy()
    pooled["label"] = pooled["model"] + " | " + pooled["feature_set"]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(pooled["label"], pooled["RMSE"])
    ax.set_xlabel("Held-out headline RMSE")
    ax.set_title("Held-out performance — frozen specification remains primary")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    selected_yearly = final_results[
        final_results["scope"].eq("year")
        & final_results["model"].eq(FROZEN_MODEL)
        & final_results["feature_set"].eq(FROZEN_FEATURE_SET)
    ].sort_values("year")
    mean_yearly = final_results[
        final_results["scope"].eq("year")
        & final_results["model"].eq("Mean")
        & final_results["feature_set"].eq("benchmark")
    ].sort_values("year")
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(selected_yearly["year"].astype(int), selected_yearly["RMSE"], marker="o", label="Frozen model")
    ax.plot(mean_yearly["year"].astype(int), mean_yearly["RMSE"], marker="o", label="Mean benchmark")
    ax.set_xlabel("Test year")
    ax.set_ylabel("RMSE")
    ax.set_title("Year-specific held-out performance")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 16. Save configuration, report tables, and fitted model artifacts

The definitive run exports the frozen primary model together with the exact feature
order, categorical levels, tuning parameters, benchmark mean, and software versions.
The native XGBoost JSON file is also saved because it is generally more portable
across XGBoost versions than a Python pickle. The `.pkl` bundle is convenient for
team members working in the same Python environment.

In [ ]:
configuration = {
    "data_path": str(DATA_PATH),
    "quick_mode": QUICK_MODE,
    "random_state": RANDOM_STATE,
    "winsor_quantiles": [WINSOR_LOWER, WINSOR_UPPER],
    "test_start_year": TEST_START_YEAR,
    "headline_test_years": HEADLINE_TEST_YEARS,
    "thin_test_years": THIN_TEST_YEARS,
    "validation_years": cv_years,
    "min_year_observations": MIN_YEAR_OBSERVATIONS,
    "screen_rows_per_year": SCREEN_ROWS_PER_YEAR,
    "tuning_rows_per_year": TUNING_ROWS_PER_YEAR,
    "final_rows_per_year": FINAL_ROWS_PER_YEAR,
    "run_optional_refits": RUN_OPTIONAL_REFITS,
    "run_linear_family_by_block_robustness": RUN_LINEAR_FAMILY_BY_BLOCK_ROBUSTNESS,
    "run_xgb_boundary_sensitivity": RUN_XGB_BOUNDARY_SENSITIVITY,
    "run_clipped_xgb_robustness": RUN_CLIPPED_XGB_ROBUSTNESS,
    "run_heldout_increment_tests": RUN_HELDOUT_INCREMENT_TESTS,
    "run_heterogeneity_analysis": RUN_HETEROGENEITY_ANALYSIS,
    "run_seen_new_firm_robustness": RUN_SEEN_NEW_FIRM_ROBUSTNESS,
    "run_macro_sensitivity": RUN_MACRO_SENSITIVITY,
    "selected_linear_family": BEST_LINEAR_FAMILY,
    "linear_block_parameters": LINEAR_BLOCK_PARAMS,
    "xgb_block_parameters": XGB_BLOCK_PARAMS,
    "frozen_model": FROZEN_MODEL,
    "frozen_feature_set": FROZEN_FEATURE_SET,
    "frozen_parameters": FROZEN_PARAMETERS,
    "feature_sets": FEATURE_SETS,
}
with open(OUTPUT_DIR / "run_configuration.json", "w", encoding="utf-8") as file:
    json.dump(configuration, file, indent=2, default=float)

if SAVE_MODELS and RUN_FINAL_TEST and fitted_artifacts:
    import platform
    import sklearn

    # Export every fitted block-specific model for reproducibility.
    for (model_name, block_name), artifact in fitted_artifacts.items():
        model = artifact["model"]
        if model_name in {"Ridge", "Lasso", "ElasticNet"}:
            joblib.dump(
                model,
                OUTPUT_DIR / f"{model_name}_{block_name}.joblib",
                compress=3,
            )
        else:
            # Native JSON is the preferred portable XGBoost representation.
            model.save_model(OUTPUT_DIR / f"xgboost_{block_name}.json")

    # Export one self-contained bundle for the frozen primary specification.
    frozen_artifact = fitted_artifacts[(FROZEN_MODEL, FROZEN_FEATURE_SET)]
    frozen_bundle = {
        "bundle_format_version": 1,
        "model_name": FROZEN_MODEL,
        "feature_set": FROZEN_FEATURE_SET,
        "feature_columns": FEATURE_SETS[FROZEN_FEATURE_SET],
        "parameters": FROZEN_PARAMETERS,
        "model": frozen_artifact["model"],
        "category_levels": frozen_artifact["category_levels"],
        "training_mean": final_mean,
        "target_column": TARGET,
        "year_column": YEAR_COL,
        "firm_id_column": ID_COL,
        "categorical_columns": [
            column for column in CATEGORY_COLS
            if column in FEATURE_SETS[FROZEN_FEATURE_SET]
        ],
        "headline_test_years": HEADLINE_TEST_YEARS,
        "software_versions": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__,
            "xgboost": xgb.__version__,
            "joblib": joblib.__version__,
        },
        "important_note": (
            "The model expects the already engineered feature columns in "
            "'feature_columns'; it does not recreate features from the raw "
            "Amadeus source data."
        ),
    }
    joblib.dump(
        frozen_bundle,
        OUTPUT_DIR / "frozen_primary_model_bundle.pkl",
        compress=3,
    )

    # Save a second native file for the frozen XGBoost model.
    if FROZEN_MODEL == "XGBoost":
        frozen_artifact["model"].save_model(
            OUTPUT_DIR / "frozen_primary_xgboost_model.json"
        )

    # Human-readable metadata can be inspected without loading a pickle.
    frozen_metadata = {
        key: value
        for key, value in frozen_bundle.items()
        if key not in {"model", "category_levels"}
    }
    with open(
        OUTPUT_DIR / "frozen_primary_model_metadata.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(frozen_metadata, file, indent=2, default=float)

print(f"Outputs written to: {OUTPUT_DIR.resolve()}")


## 17. Reload the frozen model and report outputs without retraining

After one team member completes the definitive full run and commits or shares the
`outputs/time_aware_model_report_lecture_aligned/` folder, another team member can
run only the following cell. It reloads the frozen model, the development and
held-out result tables, and the saved predictions. It can therefore reproduce
the main tables and graphs without repeating hyperparameter search or model fitting.

The model bundle is intended for trusted project files only. Python pickle/joblib
files should never be loaded from an untrusted source. For longer-term portability,
retain the native XGBoost JSON file and the JSON metadata alongside the `.pkl`
bundle.

In [ ]:
# Set to True only when reopening saved outputs after a completed definitive run.
RUN_RELOAD_DEMO = False

if RUN_RELOAD_DEMO:
    from pathlib import Path
    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    RELOAD_OUTPUT_DIR = Path("outputs/time_aware_model_report_lecture_aligned")

    required_files = [
        "frozen_primary_model_bundle.pkl",
        "development_cv_summary.csv",
        "development_cv_by_year.csv",
        "held_out_test_results.csv",
        "held_out_test_predictions.parquet",
    ]
    missing_files = [
        name for name in required_files
        if not (RELOAD_OUTPUT_DIR / name).exists()
    ]
    if missing_files:
        raise FileNotFoundError(
            "Complete the definitive full run or obtain the output folder first. "
            f"Missing: {missing_files}"
        )

    loaded_frozen_bundle = joblib.load(
        RELOAD_OUTPUT_DIR / "frozen_primary_model_bundle.pkl"
    )
    loaded_development_cv_summary = pd.read_csv(
        RELOAD_OUTPUT_DIR / "development_cv_summary.csv"
    )
    loaded_development_per_year = pd.read_csv(
        RELOAD_OUTPUT_DIR / "development_cv_by_year.csv"
    )
    loaded_final_results = pd.read_csv(
        RELOAD_OUTPUT_DIR / "held_out_test_results.csv"
    )
    loaded_final_predictions = pd.read_parquet(
        RELOAD_OUTPUT_DIR / "held_out_test_predictions.parquet"
    )

    loaded_model_name = loaded_frozen_bundle["model_name"]
    loaded_feature_set = loaded_frozen_bundle["feature_set"]

    primary_result = loaded_final_results[
        loaded_final_results["scope"].eq("pooled")
        & loaded_final_results["model"].eq(loaded_model_name)
        & loaded_final_results["feature_set"].eq(loaded_feature_set)
    ]
    print("Reloaded frozen primary result:")
    display(primary_result.round(6))

    # Development comparison.
    plot_dev = loaded_development_cv_summary[
        loaded_development_cv_summary["feature_set"].isin(
            ["firm", "firm_survey", "firm_survey_text"]
        )
    ].copy()
    plot_dev["label"] = (
        plot_dev["model"].astype(str)
        + " | "
        + plot_dev["feature_set"].astype(str)
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(plot_dev["label"], plot_dev["RMSE"])
    ax.set_xlabel("Development expanding-window RMSE")
    ax.set_title("Development-period model comparison")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    # Held-out comparison.
    pooled = loaded_final_results[
        loaded_final_results["scope"].eq("pooled")
        & loaded_final_results["feature_set"].isin(
            ["firm", "firm_survey", "firm_survey_text"]
        )
    ].copy()
    pooled["label"] = (
        pooled["model"].astype(str)
        + " | "
        + pooled["feature_set"].astype(str)
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(pooled["label"], pooled["RMSE"])
    ax.set_xlabel("Held-out RMSE")
    ax.set_title("Held-out performance comparison")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    def predict_with_loaded_frozen_model(
        engineered_frame: pd.DataFrame,
    ) -> np.ndarray:
        """Score an already feature-engineered frame with the saved model."""
        bundle = loaded_frozen_bundle
        feature_columns = bundle["feature_columns"]
        missing = [
            column for column in feature_columns
            if column not in engineered_frame.columns
        ]
        if missing:
            raise ValueError(
                "The scoring frame is missing engineered features: "
                f"{missing[:10]}"
            )

        model = bundle["model"]
        if bundle["model_name"] in {"Ridge", "Lasso", "ElasticNet"}:
            return np.asarray(
                model.predict(engineered_frame[feature_columns]),
                dtype=float,
            )

        X = engineered_frame[feature_columns].copy()
        category_levels = bundle["category_levels"] or {}
        for column, levels in category_levels.items():
            values = X[column].astype("string")
            allowed = pd.Index(levels)
            values = values.where(values.isin(allowed))
            X[column] = pd.Categorical(values, categories=allowed)
        return np.asarray(model.predict(X), dtype=float)

    print(
        "Model bundle loaded. Use predict_with_loaded_frozen_model(frame) "
        "only on data containing the same engineered feature columns."
    )


## 18. Recommended reporting sequence

1. Describe the 2010–2019 development period and the expanding-window folds for 2015–2019.
2. Explain that hyperparameter search uses firm-level subsamples but the final full-mode refit uses all development observations.
3. Report the selected regularized-linear family, its one-standard-error parameters, convergence audit, and the block-specific family robustness table.
4. Report block-specific XGBoost settings and the targeted development-only boundary sensitivity.
5. State the specification frozen on development-period evidence.
6. Present that specification's 2020–2022 held-out RMSE, MAE, and out-of-sample R² as the primary result.
7. Present alternative held-out rows, clipped-XGBoost results, and seen-versus-new-firm results only as robustness evidence, not as a basis for reselection.
8. Compare the survey and text increments across development and held-out samples; emphasize effect sizes and sign stability because the number of annual clusters is small.
9. Answer Research Question 2 with the size, leverage, age, and sector tables, describing them as exploratory heterogeneity evidence.
10. Interpret macro responsiveness using coherent historical macro-vector scenarios. Treat one-variable perturbations as synthetic predictive sensitivity, not causal effects.
11. Report the thin 2023 sample separately as a distribution-shift or sample-composition sensitivity check.
